# Quality control of genotype data: call rate, MAF, Hardy–Weinberg equilibrium and problematic samples
### *Hypolimnas misippus* — starting from the ANGSD genotype likelihoods (`HypMis_calls.beagle.gz`)

*Practical for the course "Genome-Wide Association Studies (GWAS) for Evolutionary Biology" (prstats.org). It continues `Association_ANGSD_PLINK_SNPTEST_Hypolimnas_misippus_Colab.ipynb`.*

---

## Why a QC notebook?

A GWAS finds the *tiny* number of variants whose genotype differs systematically between cases and controls out of hundreds of thousands of variants. Artefacts are far more likely than real signal to produce a systematic difference: a handful of badly genotyped SNPs, or one sample of poor quality, can generate **thousands of false associations**, or hide real ones. **Quality control (QC)** removes variants and individuals whose genotype data cannot be trusted, **before** any test is run.

The standard QC steps of GWAS (Anderson et al. 2010; Marees et al. 2018) are the topic of this notebook:

| Step | Level | Question |
|---|---|---|
| **SNP call rate** | variant | Is the variant genotyped in enough individuals — and equally well in cases and controls? |
| **Minor allele frequency (MAF)** | variant | Is the minor allele common enough to be tested reliably? |
| **Hardy–Weinberg equilibrium (HWE)** | variant | Do genotype frequencies look like those of a randomly mating population — or like genotyping errors / collapsed paralogs? |
| **Sample call rate** | individual | Is the individual's data complete enough? |
| **Heterozygosity** | individual | Too many/too few heterozygous genotypes = contamination, inbreeding, low coverage |
| **Relatedness / duplicates** | individual | Are there close relatives or accidental duplicates? |
| **Outliers in PCA** | individual | Is someone genetically very different from the rest (ancestry, species mix-up, batch)? |

## The data

This matters: with low-coverage sequencing a genotype is **uncertain**, and many classical QC statistics (computed on hard calls) become **biased**, because the genotypes that reach the confidence cutoff are a *non-random* subset (for example, depending on depth and allele frequency, heterozygotes can end up over- or under-represented, which distorts MAF and looks like an HWE violation). We will therefore compute every QC metric in two ways, from hard calls (as PLINK does) and from likelihoods (as ANGSD/PCAngsd do), and **compare them**.

## What you will get

* graphs and tables for each QC metric, for variants and for samples;
* a count of how many variants and samples fail **each** criterion, **how they overlap** (UpSet plots), and how many variants **remain** after sequential filtering;
* a comparison of two threshold sets: "standard GWAS thresholds" vs thresholds **adapted to low-coverage data**;
* a cross-check of our numbers against PLINK;
* the QC-filtered data (BEAGLE file + variant and sample lists) for further analysis.

## Notebook map

| Part | Content |
|---|---|
| 0 | Background (long): genotype likelihoods, call rate, MAF, HWE, sample QC, order of operations |
| 1 | Setup: Drive, software, parameters, input files |
| 2 | Scan the BEAGLE file: allele frequencies, posteriors, hard calls, per-SNP statistics |
| 3 | **Variant QC**: call rate, MAF, HWE, differential missingness, combined view, sweeps over thresholds |
| 4 | **Sample QC**: call rate, heterozygosity, relatedness, PCA outliers; the *final* variant filtering |
| 5 | Summary figures, comparison of threshold sets, export, PLINK cross-check |
| 6 | Discussion, pitfalls, exercises |

# Part 0 — Background

## 0.1 Genotype likelihoods, posteriors and hard calls (what is in the BEAGLE file)

For individual *i* at a biallelic site with alleles **A** (allele 1; here: ANGSD's *major* allele) and **B** (allele 2; the *minor* allele) there are three genotypes: `AA`, `AB`, `BB`.
From the reads covering the site ANGSD computes the **genotype likelihoods**

$$L_i(g)\;=\;P(\text{reads of } i \mid g),\qquad g\in\{AA,AB,BB\}$$

and writes them (scaled to sum to 1) to the BEAGLE file, three columns per individual:

```
marker              allele1 allele2   Ind0    Ind0    Ind0     Ind1    Ind1    Ind1  ...
JBBEEQ010000001.1_41   0       3     0.333   0.333   0.333    0.739   0.260   0.000 ...
                                      ^^^ individual 0 has no reads: flat likelihood
```

* **No data** → three equal values (1/3, 1/3, 1/3): *nothing is known*.
* **One read with allele A** → AA is most likely, AB less so (the read could have come from the A chromosome of a heterozygote), BB very unlikely: `0.74, 0.26, 0.00`. A **heterozygote cannot be recognised** from a single read.

To obtain a *genotype* we need a **prior**. Under Hardy–Weinberg equilibrium and allele frequency *p* of B (estimated from all individuals by an EM algorithm — the same computation ANGSD does with `-doMaf`) the prior is `(1−p)², 2p(1−p), p²` and the **posterior** is

$$P(g\mid\text{reads}) \;=\; \frac{L_i(g)\,\pi_g}{\sum_{g'} L_i(g')\,\pi_{g'}}$$

A **hard call** is the genotype with the highest posterior **if** that posterior is at least a cutoff (here 0.9, as in the association notebook). Otherwise the genotype is **missing**. This is exactly what PLINK received in the previous notebook.

Three consequences that shape this whole notebook:

1. **Two kinds of "missing".** An individual can have *no data at all* at a site (flat likelihoods), or *some data but not enough to reach the cutoff*. We call the fraction of individuals with some data the **data rate** and the fraction with a confident hard call the **call rate**. Call rate ≤ data rate.
2. **Missingness depends on the genotype.** Whether a call reaches the posterior cutoff depends on the number of reads and on which alleles they show: a heterozygote needs reads of *both* alleles (≥ 2 reads), a homozygote for a rare allele can be called from very few reads because the prior favours it, a homozygote for a common allele needs more. At low depth the hard-call data therefore **over- or under-represent heterozygotes** compared with the truth; the direction and size depend on depth, cutoff and allele frequency (the simulation in Part 2 gives an example).
3. **Missingness depends on depth**, which varies between samples and between sites (repeats, GC content, mappability). So call rate is a proxy for *quality* of both samples and sites.

## 0.2 SNP call rate (genotyping rate, missingness)

**Definition.** For SNP *s*: call rate = (number of individuals with a hard call) / (number of individuals). Missingness = 1 − call rate. (PLINK: `--geno 0.05` removes SNPs with > 5% missing, i.e. call rate < 0.95.)

**Why filter?**
* Sites with low call rate are *noisy*: their allele frequency and test statistics rest on few individuals.
* More importantly, **missingness is rarely random.** If it differs between cases and controls (*differential missingness*), the SNP can appear associated with the phenotype purely because of the pattern of missing genotypes. This is one of the commonest causes of false-positive GWAS hits.
* Low coverage regions are also enriched for mapping errors.

**Standard thresholds.** 95–99% call rate for SNPs in array-based GWAS. For **low-coverage sequencing** these thresholds would remove nearly everything (a typical call rate with 2–5× depth is 30–70%), so you choose the threshold from the *distribution* and accept a lower value, **but** you then must pay attention to differential missingness and to the *hard-call bias* (hets missing more). Using likelihood-based methods downstream avoids discarding the uncertain genotypes.

**Strategy.** A two-stage filter is recommended: a *lenient* SNP filter first (so that bad SNPs do not distort the sample QC), then the sample QC, then the *strict* SNP filter on the retained individuals. We follow this order (0.7).

## 0.3 Minor allele frequency (MAF)

**Definition.** MAF = frequency of the less common allele, between 0 and 0.5. The **minor allele count** is MAC = 2·n·MAF (n = number of individuals), the number of copies of the minor allele in the sample.

**Why filter?**
* **Power**: the information about association is proportional to *p(1−p)*. A variant with 3 minor-allele copies in 20 individuals cannot be associated with anything at a significance level of 10⁻⁸.
* **Statistical validity**: the standard tests (χ², logistic regression Wald/score tests) rely on asymptotics that fail when some genotype classes contain 0–5 individuals; the p-values are then unreliable (usually anti-conservative).
* **Error rate**: if the genotyping error rate is ε per genotype, a rare variant with true MAF < ε is dominated by errors. Most "rare" alleles in low-coverage data are sequencing errors.
* **Stability of other QC metrics**: HWE tests, relatedness and PCA are unstable with rare variants.

**Typical thresholds.** 1–5% in GWAS of a few thousand individuals; 0.1% in biobanks; with **n = 20** a MAF of 5% is only **2 allele copies**: practically nothing can be learned about such variants, so MAC-based thresholds (≥ 5–10 copies) are more meaningful than MAF. **Important:** ANGSD already applied `-minMaf 0.05` when it called SNPs, so the file contains no variant with an estimated MAF below ≈ 5%; the MAF filter below mostly removes variants whose frequency estimate shifts below the threshold after we recompute it (hard-call MAF ≠ likelihood MAF) or when you raise the threshold.

**Estimating MAF.** (a) From **hard calls**: biased, because the called genotypes are a non-random subset. (b) From **likelihoods** (EM, as ANGSD): uses all reads and is approximately unbiased. We use (b) as the primary estimate, and show (a) for comparison.

## 0.4 Hardy–Weinberg equilibrium (HWE)

### The law
G. H. Hardy and W. Weinberg (1908) showed that in an **infinitely large, randomly mating population, without selection, mutation, migration or population structure**, allele frequencies stay constant and genotype frequencies after **one generation** of random mating are fixed by the allele frequencies:

$$f(AA)=(1-p)^2,\qquad f(AB)=2p(1-p),\qquad f(BB)=p^2$$

where *p* is the frequency of allele B. The expected proportion of heterozygotes can never exceed 0.5 (reached at *p* = 0.5), and **for rare alleles nearly all copies are in heterozygotes** (`2pq ≈ 2p` versus `p²`): 5% minor allele → 9.5% heterozygotes, 0.25% homozygotes.

On the **de Finetti diagram** (we draw it below) each SNP is a point inside a triangle whose corners are the three genotypes; HWE populations lie on a **parabola**.

### Deviations: the inbreeding coefficient F
A convenient parameterisation: `f(AB) = 2p(1−p)(1−F)`. **F = 0** means HWE; **F > 0** means a **deficit** of heterozygotes (homozygote excess); **F < 0** means a **heterozygote excess**.

### Why test HWE in QC? Because *errors* cause deviations
| Observation | Typical cause | Relevance |
|---|---|---|
| **Heterozygote deficit** (F > 0) | **genotyping errors**: heterozygotes called as homozygotes (at low coverage: a typical error, but miscalls in the other direction occur as well); allele dropout/null alleles; **population structure** (Wahlund effect: pooling two sub-populations with different allele frequencies); inbreeding; selfing; assortative mating | With hard calls from low-coverage data the HWE test is **mis-calibrated** (in either direction): **test with likelihoods** |
| **Heterozygote excess** (F < 0) | **mapping errors**: reads from two paralogous loci pile up on one reference position, creating "fixed heterozygosity" (collapsed duplicates/repeats); heterozygote advantage (rare, weak); contamination; Z-linked variants in the heterogametic sex | A very characteristic and *very important* artefact for non-model organisms with fragmented assemblies: **sites with strong heterozygote excess are almost always paralogs** |
| **Both** | batch effects, mixtures of populations | |

### How to test
* **χ² goodness-of-fit** with 1 degree of freedom (3 genotype classes − 1 − 1 estimated allele frequency). Quick, but **inaccurate** when expected counts are small (rare variants, small samples).
* **Exact test** (Wigginton, Cutler & Abecasis 2005 — what PLINK `--hwe` uses): conditional on the number of A and B alleles, the number of heterozygotes has an exact distribution; the p-value sums the probabilities of all outcomes at most as likely as the one observed. Valid for any sample size.
* **Likelihood-ratio test on genotype likelihoods** (e.g. in ANGSD/PCAngsd/ngsTools): compare the maximum likelihood of the *reads* under (i) HWE (one parameter *p*) with (ii) unconstrained genotype frequencies (two parameters); `LRT = 2(logL₁ − logL₀)` ~ χ²₁. Does not need genotype calls, therefore **not biased by the low coverage**. We implement it.

### The threshold, and **where** to test
* **In controls only** for case-control studies. A real association with the disease can itself cause deviations from HWE *in cases*. Testing all individuals may remove the very variant you are looking for. Typical thresholds: **P < 10⁻⁶ in controls** (PLINK default practice), **10⁻¹⁰ in cases** (optional, more lenient to avoid removing true signals).
* The threshold is *much* stricter than 0.05 because (a) hundreds of thousands of tests are made; (b) small deviations are expected from non-random mating and structure and are not errors.
* **Power depends on sample size.** With 10 controls, an exact test cannot reject HWE even for strong departures (see the simulation in Part 3). HWE filtering is *not* meaningful for very small samples, but excess heterozygosity from paralogs is so extreme that it *is* visible in the allele-frequency–heterozygosity scatter plots.
* **Population structure** inflates F > 0 even without errors. If your sample includes several populations, test HWE within each (or after removing outliers).

## 0.5 Sample-level QC

* **Call rate (missingness) per individual.** Low-quality DNA/libraries, low coverage → many missing genotypes → unreliable. Standard threshold: remove samples with > 2–5% missing. For low-coverage sequencing *every* individual has high missingness, so use a **relative** criterion (outlier vs the rest) together with absolute depth.
* **Heterozygosity.** `F_sample = (O_hom − E_hom)/(N − E_hom)` (PLINK `--het`). *Too much* heterozygosity → contamination or sample mix-up (DNA from two individuals), or paralog-rich mapping; *too little* → inbreeding, **low coverage (hets not called!)**, allelic dropout. Remove individuals ±3 SD (or ±3 robust SD) from the mean. **Warning**: for low-coverage data heterozygosity can correlate *strongly* with depth. We therefore check the relationship and flag individuals relative to the depth-expected value.
* **Relatedness and duplicates.** Standard association tests assume **independent** individuals. Duplicates, siblings or parents/offspring share more alleles than assumed, inflating test statistics (false positives) and distorting PCA. From the **genomic relationship matrix (GRM)** we estimate the **kinship coefficient φ** (= GRM/2) for each pair: **0.5 = identical** (duplicate, monozygotic twin or clone), **0.25 = first-degree** relatives (parent–offspring, full sibs), **0.125 = second-degree**, 0.0625 third-degree, ≈ 0 unrelated. The conventional cut-offs (KING) are the midpoints: **φ > 0.354 duplicate / MZ, 0.177–0.354 first-degree, 0.088–0.177 second-degree**. Remove one individual from each related pair (the one with lower data quality).
* **PCA outliers.** The first PCs of the genotype matrix describe ancestry. An individual far from the cloud is of different ancestry (another population or species), may be contaminated, or comes from a different batch. EIGENSOFT's rule: remove individuals > 6 standard deviations from the mean on any of the top PCs. Population stratification also affects association tests and is handled with PC covariates (see the association notebook).
* **Sex check** (not done here). In species with sex chromosomes the heterozygosity of the Z/X chromosome identifies sex errors; our data are autosomal.

## 0.6 Order of operations and reporting

QC steps interact, so the **order matters** (PLINK tutorial, Marees et al. 2018):

1. **Lenient variant filter** – remove SNPs that are genotyped in very few individuals (e.g. < 20%): they would distort the sample QC.
2. **Sample QC** – on the remaining SNPs: call rate, heterozygosity, relatedness, PCA.
3. **Strict variant filter** – on the *retained* individuals: call rate, differential missingness, MAF, HWE (statistics change when individuals are removed, so they are **recomputed**).

Also: (i) **record every threshold and every count** (the funnel table at the end is the “QC paragraph” of a paper); (ii) *do not blindly remove* outlier individuals: understand why they are outliers; (iii) test the **sensitivity** of the result to the thresholds (we do this: sweeps in Part 3).

## 0.7 What the numbers will look like, and why low coverage is special

| Statistic | High-coverage array/WGS | **Low-coverage hard calls** |
|---|---|---|
| SNP call rate | > 98% | 20–80% (depends on depth) |
| MAF estimate | accurate | biased by selective calling; use likelihoods |
| HWE | few SNPs fail | **test is mis-calibrated** (too many or too few failures) because of selective calling |
| Sample heterozygosity | tight | tracks depth |

The best remedy is to **keep uncertainty** (genotype likelihoods or dosages) for downstream analysis. QC is still needed: it removes paralogs, badly covered sites and bad samples.

# Part 1 — Setup

This notebook needs no compilers: it uses Python (numpy, pandas, scipy, matplotlib, seaborn), which Colab already provides. We mount Google Drive, and install **PLINK 1.9** (via `apt`) only for the cross-check in Part 5. If `apt` fails the cross-check is skipped.

In [ ]:
# 1.1  Mount Drive, install PLINK (optional)
from google.colab import drive
drive.mount("/content/drive")
import subprocess, shutil
r = subprocess.run("apt-get -qq update && apt-get -qq install -y plink1.9", shell=True, capture_output=True, text=True)
PLINK = shutil.which("plink1.9") or shutil.which("plink")
print("PLINK:", PLINK or "not available (the cross-check in Part 5 will be skipped)")

## 1.2 Parameters

All thresholds are collected here. Two **threshold sets** are defined, applied to the same data and compared at the end:

* **`standard`** – the usual array-based GWAS thresholds (PLINK-style): SNP call rate 0.95, MAF 0.01, HWE P < 10⁻⁶ in controls (exact test on hard calls), sample call rate 0.95.
* **`adapted`** – thresholds suited to low-coverage data: a lower call-rate cut-off, a minor-allele-**count** criterion, **likelihood-based HWE test** (valid under uncertain genotypes), and an automatic outlier rule for samples.

`ACTIVE` selects which set is used for the final filtered data. Everything else (graphs, tables) shows both.

In [ ]:
# 1.2  Parameters
import os, re, gzip, glob, json, time, itertools, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from IPython.display import display
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

DRIVE_ROOT = Path("/content/drive/MyDrive")
BEAGLE_OVERRIDE = None        # e.g. "/content/drive/MyDrive/GWAS/association_HypMis/angsd/HypMis_calls.beagle.gz"

# ---- how a hard call is made (as in the association notebook)
POST_CUTOFF = 0.90            # genotype called if the best posterior is >= this
EM_ITER = 40                  # EM iterations for allele frequencies / HWE likelihoods
CHUNK = 20_000                # SNPs processed at a time (memory)
CASE_MORPH, CONTROL_MORPH = "m", "i"      # controls are used for HWE (as in a case-control GWAS)

# ---- the two threshold sets
PRESETS = {
    "standard": dict(snp_call=0.95, maf=0.01, mac=0,  hwe_p=1e-6, hwe_test="exact_hard", hwe_group="controls", diffmiss_p=1e-5,
                     sample_call=0.95, het_sd=3.0, het_adjust=False, kin=0.177, pca_sd=6.0, pca_robust=False),
    "adapted":  dict(snp_call=0.50, maf=0.05, mac=5,  hwe_p=1e-4, hwe_test="lrt_gl",     hwe_group="all",      diffmiss_p=1e-3,
                     sample_call=None, het_sd=3.0, het_adjust=True,  kin=0.354, pca_sd=6.0, pca_robust=True),
}
ACTIVE = "adapted"            # preset used for the final filtered files
LENIENT_SNP_CALL = 0.20       # step 1 of the QC order: SNPs genotyped in fewer than this fraction are removed before the sample QC
SAMPLE_CALL_AUTO_MADS = 3.0   # 'adapted' sample call-rate rule: median − k·MAD (robust outlier rule)
THIN_KB = 10                  # distance thinning for PCA/GRM (kb): one SNP per window (reduces LD)
MAX_PCA_SNPS = 200_000
SEED = 1

# ---- colours (colour-blind safe)
C = {"case": "#4c72b0", "ctrl": "#dd8452", "pass": "#2a9d8f", "fail": "#d1495b", "grey": "#8d99ae", "dark": "#2b2d42", "gold": "#e9c46a"}
sns.set_theme(context="notebook", style="whitegrid", font_scale=0.95)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
rng = np.random.default_rng(SEED)

## 1.3 Locate the input files

The association notebook created, inside `association_HypMis/angsd/`:

* `HypMis_calls.beagle.gz` – genotype likelihoods (BEAGLE format);
* `HypMis_calls.mafs.gz` – ANGSD allele-frequency estimates (we use them to check our own EM);
* `samples_in_order.txt` – the sample IDs **in the column order of the BEAGLE file**;
* in the project folder `phenotypes.tsv` (sample, morph; maybe depth).

The order of samples is critical: individual *k* in the BEAGLE file is line *k* of `samples_in_order.txt`. We check the numbers of samples against the file header to catch outdated files (an earlier run with a different sample list).

In [ ]:
# 1.3  Find the files on Drive
def find_on_drive(name, kind="f", maxdepth=7):
    res = subprocess.run(["find", str(DRIVE_ROOT), "-maxdepth", str(maxdepth), "-type", kind, "-name", name], capture_output=True, text=True)
    return [Path(p) for p in res.stdout.splitlines() if p.strip()]

if BEAGLE_OVERRIDE: BEAGLE = Path(BEAGLE_OVERRIDE)
else:
    hits = find_on_drive("HypMis_calls.beagle.gz")
    assert hits, "HypMis_calls.beagle.gz not found on Drive. Run the association notebook (ANGSD calling section) first or set BEAGLE_OVERRIDE."
    hits.sort(key=lambda p: -p.stat().st_mtime)               # newest first
    if len(hits) > 1: print("several files found, using the newest:", *hits, sep="\n  ")
    BEAGLE = hits[0]
ANGSD_DIR = BEAGLE.parent
MAFS = ANGSD_DIR / "HypMis_calls.mafs.gz"
OUT = ANGSD_DIR.parent / "qc"; OUT.mkdir(exist_ok=True)
PROJECT_DIR = ANGSD_DIR.parent.parent
print("BEAGLE :", BEAGLE, f"({BEAGLE.stat().st_size/1e6:.1f} MB)\nMAFS   :", MAFS, MAFS.exists(), "\nOutput :", OUT)

with gzip.open(BEAGLE, "rt") as fh: header = fh.readline().rstrip("\n").split("\t")
n_cols = len(header) - 3
assert n_cols % 3 == 0, "unexpected BEAGLE header"
N_FILE = n_cols // 3
so = ANGSD_DIR / "samples_in_order.txt"
SAMPLES = [l.strip() for l in so.read_text().splitlines() if l.strip()] if so.exists() else None
assert SAMPLES is not None, f"{so} not found"
assert len(SAMPLES) == N_FILE, (f"The BEAGLE file has {N_FILE} individuals but samples_in_order.txt lists {len(SAMPLES)}. "
                                "One of the two is outdated (e.g. ANGSD was run with a different BAM list): re-run ANGSD with FORCE=True.")
N = N_FILE
print(f"{N} individuals in the BEAGLE file; header: {header[:6]} ...")

In [ ]:
# 1.4  Sample table: morph (case/control) and optional covariates
ph = PROJECT_DIR / "phenotypes.tsv"
if not ph.exists():
    hits = find_on_drive("phenotypes.tsv"); ph = hits[0] if hits else None
if ph is not None and ph.exists():
    pheno = pd.read_csv(ph, sep="\t")
else:
    mh = find_on_drive("Hypolimnas_misippus_CAMID_morph.txt"); assert mh, "no phenotypes.tsv and no morph file found"
    pheno = pd.read_csv(mh[0], sep="\t"); pheno.columns = ["sampleID", "morph"]
pheno = pd.DataFrame({"sampleID": SAMPLES}).merge(pheno, on="sampleID", how="left")
assert pheno.morph.notna().all(), "some samples have no morph annotation"
pheno["is_case"] = (pheno.morph == CASE_MORPH).values
pheno["is_ctrl"] = (pheno.morph == CONTROL_MORPH).values
IS_CASE, IS_CTRL = pheno.is_case.values, pheno.is_ctrl.values
print(f"{N} samples: {IS_CASE.sum()} cases ({CASE_MORPH}), {IS_CTRL.sum()} controls ({CONTROL_MORPH})")
display(pheno.head())
if N < 50:
    print(f"\n⚠ Only {N} individuals: all tests below have LOW POWER (see the HWE power simulation in Part 3). "
          "The workflow is the same for hundreds of individuals, but interpret p-values of this small data set with care.")

In [ ]:
# 1.5  Small helpers
def savefig(name):
    plt.savefig(OUT / f"{name}.png", dpi=150, bbox_inches="tight")

def fmt(n): return f"{int(n):,}"

def panel_label(ax, s):
    ax.text(-0.12, 1.06, s, transform=ax.transAxes, fontsize=13, fontweight="bold")

def mad_z(x):
    x = np.asarray(x, float); m = np.nanmedian(x); mad = np.nanmedian(np.abs(x - m)) * 1.4826
    return (x - m) / (mad if mad > 0 else np.nanstd(x) + 1e-12)
print("ready")

# Part 2 — Scan the BEAGLE file: from likelihoods to QC statistics

We read the BEAGLE file in **chunks of SNPs** (memory!) and for every SNP compute, **with all individuals** (the first scan), or **with the retained individuals** (the final scan, after the sample QC):

1. **Allele frequency by EM** from the likelihoods (the same model as ANGSD `-doMaf`): in each iteration the posterior of each genotype is computed with the current HWE prior, and the frequency is updated as the mean expected number of B alleles (individuals without data are ignored).
2. **Posteriors and hard calls** (cutoff `POST_CUTOFF`): *called* = best posterior ≥ cutoff; the hard call is 0, 1 or 2 copies of allele B; −1 = missing.
3. **Data rate and call rate** of the SNP, overall and separately in cases and controls.
4. **HWE likelihood-ratio test** from the likelihoods (all individuals, and controls only), including the inbreeding coefficient F.
5. **Hard-call genotype counts** (all, controls, cases), from which exact HWE tests, hard-call MAF, and tests of differential missingness are derived.
6. A **thinned set of SNPs** (one per `THIN_KB` kb, common and well covered) for which the posterior **dosage** (expected number of B alleles, 0–2) is kept: it will feed the relatedness matrix and the PCA in the sample QC.

### The EM algorithm in four lines

For SNP *s* with likelihoods L_i(g), and frequency *p* (of B):

```
repeat:
    π = [(1-p)², 2p(1-p), p²]                       # HWE prior
    posterior_i(g) ∝ L_i(g)·π_g                      # for each individual with data
    p ← (1/2n) Σ_i [ posterior_i(AB) + 2·posterior_i(BB) ]
```
For the **HWE test** the same idea is run a second time with *unconstrained* genotype frequencies `f = (f_AA, f_AB, f_BB)`: `f ← mean_i posterior_i(·)` where the prior is `f` itself. The two log-likelihoods `log Π_i Σ_g L_i(g)·prior_g` are compared: `LRT = 2(logL_free − logL_HWE)`, which under HWE follows χ² with 1 d.f.

In [ ]:
# 2.1  Core functions: EM allele frequency, HWE likelihood-ratio test, exact test
def normalise_L(X):
    X = np.nan_to_num(X, nan=1/3, posinf=1/3, neginf=1/3)
    s = X.sum(-1, keepdims=True)
    return np.where(s > 0, X / np.where(s > 0, s, 1), 1/3)

def has_data_mask(L, tol=1e-3):
    return (L.max(-1) - L.min(-1)) > tol                    # flat likelihood (1/3,1/3,1/3) = individual without reads

def hwe_prior(p):
    q = 1 - p
    return np.stack([q * q, 2 * p * q, p * p], -1)

def em_p(L, valid, iters, p0=0.2):
    # allele frequency of allele 2 (B) by EM, ignoring individuals without data
    m = L.shape[0]; p = np.full(m, p0); nv = valid.sum(1); w = valid.astype(float)
    for _ in range(iters):
        post = L * hwe_prior(p)[:, None, :]
        post /= post.sum(-1, keepdims=True) + 1e-300
        p = (((post[..., 1] + 2 * post[..., 2]) * w).sum(1)) / (2 * np.maximum(nv, 1))
        p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.where(nv > 0, p, np.nan)

def loglik(L, valid, f):
    mix = (L * f[:, None, :]).sum(-1)
    return (np.log(np.maximum(mix, 1e-300)) * valid).sum(1)

def hwe_lrt(L, valid, iters, p=None):
    # LRT of HWE from genotype likelihoods: returns (LRT, F, p). F = 1 - f_AB / (2pq) from the unconstrained fit
    m = L.shape[0]; nv = np.maximum(valid.sum(1), 1)
    if p is None: p = em_p(L, valid, iters)
    p0 = np.nan_to_num(p, nan=0.5)
    f = hwe_prior(p0); ll0 = loglik(L, valid, f)
    for _ in range(2 * iters):
        post = L * f[:, None, :]
        post /= post.sum(-1, keepdims=True) + 1e-300
        f = (post * valid[..., None]).sum(1) / nv[:, None]
        f = np.maximum(f, 1e-9); f /= f.sum(1, keepdims=True)
    ll1 = loglik(L, valid, f)
    lrt = np.maximum(2 * (ll1 - ll0), 0)
    pf = f[:, 2] + f[:, 1] / 2; den = 2 * pf * (1 - pf)
    F = np.where(den > 1e-6, 1 - f[:, 1] / np.maximum(den, 1e-12), np.nan)
    lrt[~(valid.sum(1) > 1)] = np.nan
    return lrt, F, p

def hwe_exact(n_het, n_hom1, n_hom2):
    # Wigginton, Cutler & Abecasis (2005) exact test of HWE (two-sided, as PLINK --hwe)
    n_het, n_hom1, n_hom2 = int(n_het), int(n_hom1), int(n_hom2)
    n = n_het + n_hom1 + n_hom2
    if n == 0: return np.nan
    hom_r, hom_c = min(n_hom1, n_hom2), max(n_hom1, n_hom2)
    rare = 2 * hom_r + n_het
    if rare == 0: return 1.0
    probs = np.zeros(rare + 1)
    mid = int(rare * (2 * n - rare) / (2 * n))
    if (mid % 2) != (rare % 2): mid += 1
    probs[mid] = 1.0; total = 1.0
    h, hr, hc = mid, (rare - mid) // 2, n - mid - (rare - mid) // 2
    while h >= 2:
        probs[h - 2] = probs[h] * h * (h - 1.0) / (4.0 * (hr + 1.0) * (hc + 1.0))
        total += probs[h - 2]; h -= 2; hr += 1; hc += 1
    h, hr, hc = mid, (rare - mid) // 2, n - mid - (rare - mid) // 2
    while h <= rare - 2:
        probs[h + 2] = probs[h] * 4.0 * hr * hc / ((h + 2.0) * (h + 1.0))
        total += probs[h + 2]; h += 2; hr -= 1; hc -= 1
    probs /= total
    return float(min(1.0, probs[probs <= probs[n_het] * (1 + 1e-7)].sum()))

def exact_p_for_counts(c0, c1, c2):
    # vectorised over SNPs through the unique count triples (very few distinct triples occur)
    arr = np.stack([c0, c1, c2], 1).astype(np.int64)
    u, inv = np.unique(arr, axis=0, return_inverse=True)
    pu = np.array([hwe_exact(t[1], t[0], t[2]) for t in u])
    return pu[inv.ravel()]

# --- unit tests of the exact test against a brute-force computation of the conditional distribution
from math import lgamma, exp, log
def brute(n_het, n_aa, n_bb):
    n = n_het + n_aa + n_bb; nb = 2 * n_bb + n_het; na = 2 * n - nb
    def lp(h):
        b = (nb - h) // 2; a = n - h - b
        return lgamma(n + 1) - lgamma(a + 1) - lgamma(h + 1) - lgamma(b + 1) + h * log(2) + lgamma(na + 1) + lgamma(nb + 1) - lgamma(2 * n + 1)
    hs = [h for h in range(nb % 2, min(na, nb) + 1, 2)]
    pr = {h: exp(lp(h)) for h in hs}
    return sum(v for v in pr.values() if v <= pr[n_het] * (1 + 1e-7))
for t in [(0, 8, 2), (5, 3, 2), (10, 5, 5), (1, 15, 4), (0, 20, 0), (12, 4, 4)]:
    assert abs(hwe_exact(*t) - brute(*t)) < 1e-9, t
print("exact HWE test verified against brute-force enumeration on test cases")
print("example: 0 heterozygotes among 8 AA / 2 BB (excess homozygotes):  P =", round(hwe_exact(0, 8, 2), 4),
      "|  12 hets, 4 AA, 4 BB:  P =", round(hwe_exact(12, 4, 4), 4))

Quick sanity check of the likelihood-based HWE test on **simulated** data: we simulate genotypes in HWE, and with excess heterozygotes (F = −0.3, the most extreme excess possible at this allele frequency is F = −0.43; like paralogs), and a heterozygote deficit (F = +0.5, like population structure or inbreeding), at a depth of ≈ 3×, produce GLs with a simple sequencing-error model, and run our LRT. The test statistic should have the right behaviour (uniform P under HWE; small P for strong deviations).

In [ ]:
# 2.2  Simulation: does the GL-based HWE test behave?
def simulate_gl(n_ind, p, F, depth, m, err=0.01, seed=0):
    r = np.random.default_rng(seed)
    f = np.array([(1 - p)**2 + F * p * (1 - p), 2 * p * (1 - p) * (1 - F), p**2 + F * p * (1 - p)])
    g = r.choice(3, size=(m, n_ind), p=f)                        # true genotypes (0,1,2 copies of B)
    cov = r.poisson(depth, size=(m, n_ind))
    nb = np.where(g == 0, r.binomial(cov, err), np.where(g == 2, r.binomial(cov, 1 - err), r.binomial(cov, 0.5)))
    na = cov - nb
    lik = np.stack([((1 - err)**na) * (err**nb), (0.5**cov) * np.ones_like(na, float), (err**na) * ((1 - err)**nb)], -1)
    lik = lik / lik.sum(-1, keepdims=True)
    return lik, g
res = {}
for name, F in [("HWE (F=0)", 0.0), ("het. excess (F=-0.3)", -0.3), ("het. deficit (F=+0.5)", 0.5)]:
    L, g = simulate_gl(n_ind=60, p=0.3, F=F, depth=3, m=2000, seed=11)
    valid = has_data_mask(L)
    lrt, Fhat, _ = hwe_lrt(L, valid, EM_ITER)
    P = stats.chi2.sf(lrt, 1)
    # same genotypes called from the reads with the 0.9 rule, tested with the exact test
    p_em = em_p(L, valid, EM_ITER); post = L * hwe_prior(p_em)[:, None, :]; post /= post.sum(-1, keepdims=True)
    called = (post.max(-1) >= POST_CUTOFF) & valid; gh = np.where(called, post.argmax(-1), -1)
    cnt = [(gh == k).sum(1) for k in range(3)]
    Pe = exact_p_for_counts(cnt[0], cnt[1], cnt[2])
    res[name] = dict(P_lrt=P, P_exact=Pe, F=Fhat)
    print(f"{name:24s}  GL-LRT: {100*np.mean(P<0.01):5.1f}% of SNPs have P<0.01   |  exact test on hard calls: {100*np.mean(Pe<0.01):5.1f}%   |  mean F-hat = {np.nanmean(Fhat):+.2f}   |  hard-call het rate = {np.mean(gh[called]==1):.2f} (truth {2*0.3*0.7*(1-F):.2f})")

**What to look for in this simulation (and why we use both tests)**

* Under **HWE** (first row) a calibrated test has about 1% of SNPs with P < 0.01. Compare the two columns: the **likelihood-based** test is close to nominal, the **hard-call exact test** is not (more false alarms).
* For the **excess** and **deficit** scenarios compare the power of the two tests and the **hard-call heterozygote rate with the true one** (printed at the end of each line). Hard-calling distorts the genotype frequencies: with the default settings (depth 3×, cutoff 0.9) heterozygotes are *over*-represented among confident calls (a heterozygote with one read of each allele is certain, while a homozygote needs several reads before its posterior passes 0.9). The sign of the bias depends on depth, cutoff and allele frequency; what matters is that it exists, and that it can **hide** a real heterozygote deficit (very low power of the exact test in the last row) or create false alarms.
* The **likelihood-based** test uses all reads, so it does not depend on which genotypes happened to be confident, and its F-hat is close to the truth.

Now the real data.

In [ ]:
# 2.3  The scan itself: one pass over the BEAGLE file
ALLELES = np.array(list("ACGT"))

def scan(keep_idx, label):
    keep_idx = np.asarray(keep_idx); Nk = len(keep_idx)
    ctrl, case = IS_CTRL[keep_idx], IS_CASE[keep_idx]
    cols = {k: [] for k in ["contig", "pos", "a1", "a2", "p", "n_data", "n_called", "n_data_case", "n_data_ctrl", "n_called_case", "n_called_ctrl",
                            "lrt_all", "F_all", "lrt_ctrl", "F_ctrl"]}
    cnt = {k: [] for k in ["all", "ctrl", "case"]}
    Gs, Ds, thin_rows, DS = [], [], [], []
    last = {"contig": None, "pos": -10**12}; row0 = 0
    t0 = time.time()
    reader = pd.read_csv(BEAGLE, sep="\t", chunksize=CHUNK, dtype={"marker": str})
    for ci, ch in enumerate(reader):
        m = len(ch)
        L = normalise_L(ch.iloc[:, 3:].to_numpy(np.float64).reshape(m, N_FILE, 3)[:, keep_idx, :])
        valid = has_data_mask(L)
        p = em_p(L, valid, EM_ITER)
        post = L * hwe_prior(np.nan_to_num(p, nan=0.5))[:, None, :]
        post /= post.sum(-1, keepdims=True) + 1e-300
        called = valid & (post.max(-1) >= POST_CUTOFF)
        G = np.where(called, post.argmax(-1), -1).astype(np.int8)
        lrt_a, F_a, _ = hwe_lrt(L, valid, EM_ITER, p=p)
        lrt_c, F_c, _ = hwe_lrt(L, valid & ctrl[None, :], EM_ITER)
        mk = ch["marker"].str.rsplit("_", n=1, expand=True)
        contig, pos = mk[0].to_numpy(), mk[1].astype(np.int64).to_numpy()
        cols["contig"].append(contig); cols["pos"].append(pos)
        cols["a1"].append(ALLELES[ch["allele1"].to_numpy()]); cols["a2"].append(ALLELES[ch["allele2"].to_numpy()])
        cols["p"].append(p); cols["n_data"].append(valid.sum(1)); cols["n_called"].append(called.sum(1))
        cols["n_data_case"].append((valid & case).sum(1)); cols["n_data_ctrl"].append((valid & ctrl).sum(1))
        cols["n_called_case"].append((called & case).sum(1)); cols["n_called_ctrl"].append((called & ctrl).sum(1))
        cols["lrt_all"].append(lrt_a); cols["F_all"].append(F_a); cols["lrt_ctrl"].append(lrt_c); cols["F_ctrl"].append(F_c)
        for k, msk in (("all", np.ones(Nk, bool)), ("ctrl", ctrl), ("case", case)):
            cnt[k].append(np.stack([((G == g) & msk).sum(1) for g in range(3)], 1))
        Gs.append(G); Ds.append(valid)
        # thinned set for PCA / relatedness
        ok = (np.minimum(p, 1 - p) >= 0.05) & (valid.sum(1) >= 0.5 * Nk)
        ds = (post[..., 1] + 2 * post[..., 2]).astype(np.float32)
        for j in np.where(ok)[0]:
            if contig[j] != last["contig"] or pos[j] - last["pos"] >= THIN_KB * 1000:
                thin_rows.append(row0 + j); DS.append(ds[j]); last = {"contig": contig[j], "pos": pos[j]}
        row0 += m
        if ci % 5 == 0: print(f"  [{label}] {row0:>9,} SNPs scanned ({time.time()-t0:.0f} s)", end="\r")
    print(f"  [{label}] {row0:,} SNPs scanned in {time.time()-t0:.0f} s          ")
    S = {k: np.concatenate(v) for k, v in cols.items()}
    S["cnt"] = {k: np.concatenate(v) for k, v in cnt.items()}
    S["G"] = np.vstack(Gs); S["valid"] = np.vstack(Ds)
    S["thin_rows"] = np.array(thin_rows, int); S["DS"] = np.array(DS, np.float32)
    S["keep_idx"] = keep_idx
    return S

import pickle
CACHE1 = OUT / "scan_all_samples.pkl"
key = (str(BEAGLE), BEAGLE.stat().st_size, N, POST_CUTOFF, EM_ITER, THIN_KB)
S1 = None
if CACHE1.exists():
    try:
        k, S1 = pickle.load(open(CACHE1, "rb"))
        if k != key: S1 = None
    except Exception: S1 = None
if S1 is None:
    S1 = scan(np.arange(N), "all samples")
    pickle.dump((key, S1), open(CACHE1, "wb"))
else:
    print("scan of all samples loaded from cache:", CACHE1.name)
M = len(S1["pos"])
print(f"{M:,} SNPs × {N} individuals;  thinned set for PCA/GRM: {len(S1['thin_rows']):,} SNPs")

## 2.4 Build the per-SNP table

From the scan we derive, for each SNP:

| Column | Meaning |
|---|---|
| `maf` | minor allele frequency from the **likelihood EM** (`min(p, 1−p)`) |
| `mac` | minor allele count: `2 × n_data × maf` |
| `maf_hard` | MAF from the hard calls only |
| `data_rate`, `call_rate` | fraction of individuals with reads / with a confident hard call |
| `call_rate_case`, `call_rate_ctrl` | the same in the two groups |
| `p_diffmiss` | Fisher exact test: is the proportion of missing hard calls different between cases and controls? |
| `lrt_p_all`, `lrt_p_ctrl`, `F_all`, `F_ctrl` | likelihood-based HWE test (P-value from χ²₁) and inbreeding coefficient, all individuals / controls only |
| `exact_p_all`, `exact_p_ctrl` | exact HWE test on the hard calls |
| `F_hard` | inbreeding coefficient from hard-call counts |

In [ ]:
# 2.4  Per-SNP table
def snp_table(S):
    n = len(S["pos"]); keep = S["keep_idx"]; Nk = len(keep)
    nc, nk = IS_CASE[keep].sum(), IS_CTRL[keep].sum()
    T = pd.DataFrame({"contig": S["contig"], "pos": S["pos"], "a1": S["a1"], "a2": S["a2"], "p": S["p"]})
    T["marker"] = T.contig + "_" + T.pos.astype(str)
    T["maf"] = np.minimum(T.p, 1 - T.p)
    T["n_data"], T["n_called"] = S["n_data"], S["n_called"]
    T["mac"] = 2 * T.n_data * T.maf
    T["data_rate"] = T.n_data / Nk; T["call_rate"] = T.n_called / Nk
    T["call_rate_case"] = S["n_called_case"] / max(nc, 1); T["call_rate_ctrl"] = S["n_called_ctrl"] / max(nk, 1)
    c = S["cnt"]["all"]; T["n_het_hard"] = c[:, 1]
    tot = np.maximum(c.sum(1), 1); pb = (c[:, 1] + 2 * c[:, 2]) / (2 * tot)
    T["maf_hard"] = np.minimum(pb, 1 - pb)
    den = 2 * pb * (1 - pb); T["F_hard"] = np.where(den > 0, 1 - (c[:, 1] / tot) / np.maximum(den, 1e-12), np.nan)
    T["het_hard_obs"] = c[:, 1] / tot
    for k in ("all", "ctrl"):
        T[f"F_{k}"] = S[f"F_{k}"]; T[f"lrt_p_{k}"] = stats.chi2.sf(S[f"lrt_{k}"], 1)
        cc = S["cnt"][k]; T[f"exact_p_{k}"] = exact_p_for_counts(cc[:, 0], cc[:, 1], cc[:, 2])
    # differential missingness (hard calls): Fisher test on the 2x2 table called/not called × case/control
    a, b = S["n_called_case"], nc - S["n_called_case"]; cc_, d = S["n_called_ctrl"], nk - S["n_called_ctrl"]
    arr = np.stack([a, b, cc_, d], 1); u, inv = np.unique(arr, axis=0, return_inverse=True)
    pu = np.array([stats.fisher_exact([[t[0], t[1]], [t[2], t[3]]])[1] for t in u])
    T["p_diffmiss"] = pu[inv.ravel()]
    return T

snp1 = snp_table(S1)
display(snp1.drop(columns=["marker"]).head())
print(f"{len(snp1):,} SNPs on {snp1.contig.nunique()} sequences")

## 2.5 Did our EM reproduce ANGSD's allele frequencies?

Before trusting any downstream number we compare our EM allele frequency with the one in ANGSD's `.mafs.gz` file (`knownEM`, the frequency of the minor allele = allele 2 of the BEAGLE file). They should agree to ≈ 3 decimals; a small remaining difference arises from a different convergence criterion.

In [ ]:
# 2.5  Validation against ANGSD .mafs
if MAFS.exists():
    mafs = pd.read_csv(MAFS, sep="\t"); mafs = mafs.loc[:, ~mafs.columns.str.startswith("Unnamed")]
    fcol = "knownEM" if "knownEM" in mafs.columns else [c for c in mafs.columns if "EM" in c][0]
    mm = snp1[["contig", "pos", "p", "a1", "a2"]].merge(mafs.rename(columns={"chromo": "contig", "position": "pos"})[["contig", "pos", fcol, "major", "minor"]], on=["contig", "pos"], how="inner")
    print(f"{len(mm):,} of {len(snp1):,} SNPs found in the .mafs file")
    r1, r2 = np.corrcoef(mm.p, mm[fcol])[0, 1], np.corrcoef(1 - mm.p, mm[fcol])[0, 1]
    print(f"correlation of our p (freq. of allele 2) with ANGSD knownEM: {r1:.4f}   (with 1-p: {r2:.4f})")
    print(f"alleles agree with ANGSD's major/minor: {np.mean((mm.a1 == mm.major) & (mm.a2 == mm.minor)):.1%}")
    print(f"mean absolute difference: {np.mean(np.abs(mm.p - mm[fcol])):.4f}")
    fig, ax = plt.subplots(figsize=(4.2, 4))
    ax.scatter(mm[fcol], mm.p, s=3, alpha=.25, color=C["case"], rasterized=True); ax.plot([0, 1], [0, 1], "k--", lw=.8)
    ax.set_xlabel("ANGSD .mafs  knownEM"); ax.set_ylabel("our EM (allele 2 frequency)"); ax.set_title("Allele frequency check")
    plt.tight_layout(); plt.show()
    if r1 < 0.9 and r2 > 0.9: print("⚠ allele coding is swapped (allele 1 is the minor allele in this file); frequencies refer to allele 2 here")
else:
    print(".mafs file not found: skipping the validation")

## 2.6 First look at the data: who has data where?

Each row is an individual, each column a SNP (random subset); colours show what we know about the genotype:

* **light grey** — no reads at all (flat likelihood),
* **yellow** — some reads but the best posterior is below the cutoff (**uncertain**; becomes missing in hard calls),
* **blue shades** — confident hard call: homozygous allele 1 / heterozygous / homozygous allele 2.

Individuals are sorted by data rate. Rows with little colour are low-coverage; columns that are grey across all rows are SNPs that nobody covered well.

In [ ]:
# 2.6  Data availability heatmap
nshow = min(400, M); cols_i = np.sort(rng.choice(M, nshow, replace=False))
G, V = S1["G"][cols_i], S1["valid"][cols_i]
code_m = np.where(G >= 0, 2 + G, np.where(V, 1, 0))                    # 0 none, 1 uncertain, 2/3/4 genotypes
order = np.argsort(-(code_m > 0).mean(0))
cmap = mpl.colors.ListedColormap(["#e9ecef", "#e9c46a", "#4c72b0", "#2a9d8f", "#d1495b"])
fig, ax = plt.subplots(figsize=(11, 0.28 * N + 2))
ax.imshow(code_m[:, order].T, aspect="auto", cmap=cmap, vmin=-0.5, vmax=4.5, interpolation="nearest")
ax.set_yticks(range(N)); ax.set_yticklabels([f"{SAMPLES[i]} ({pheno.morph[i]})" for i in order], fontsize=7); ax.set_xlabel(f"{nshow} random SNPs"); ax.grid(False)
ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(cmap.colors, ["no data", "uncertain (posterior < cutoff)", "hom. allele 1 (AA)", "heterozygote (AB)", "hom. allele 2 (BB)"])],
          loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=5, frameon=False)
ax.set_title("Genotype information per individual and SNP"); plt.tight_layout(); savefig("data_availability_heatmap"); plt.show()
print(f"overall: {100*(code_m==0).mean():.1f}% no data, {100*(code_m==1).mean():.1f}% uncertain, {100*(code_m>=2).mean():.1f}% confident hard calls")

# Part 3 — Variant QC

In this part all individuals are included. We look at each criterion separately, then combine them. Nothing is removed yet: we only **flag**.

## 3.1 SNP call rate and data rate

* **Data rate** = fraction of individuals with *any* reads at the site (flat likelihood otherwise).
* **Call rate** = fraction with a *confident* hard call (best posterior ≥ `POST_CUTOFF`).

The gap between the two is the information that is thrown away by hard-calling. We look at the distributions overall, and **separately for cases and controls**: a SNP covered better in one group than the other is suspicious.

In [ ]:
# 3.1  Call rate and data rate of SNPs
fig, ax = plt.subplots(1, 4, figsize=(19, 4))
ax[0].hist(snp1.data_rate, bins=np.linspace(0, 1, 41), color=C["grey"], alpha=.8, label="data rate")
ax[0].hist(snp1.call_rate, bins=np.linspace(0, 1, 41), color=C["case"], alpha=.8, label="call rate (hard calls)")
for nm, col in (("standard", C["fail"]), ("adapted", C["gold"])):
    ax[0].axvline(PRESETS[nm]["snp_call"], color=col, ls="--", label=f"{nm} threshold ({PRESETS[nm]['snp_call']})")
ax[0].set_xlabel("fraction of individuals"); ax[0].set_ylabel("SNPs"); ax[0].set_title("Per-SNP data rate and call rate"); ax[0].legend(fontsize=7)
panel_label(ax[0], "A")

hb = ax[1].hexbin(snp1.maf, snp1.call_rate, gridsize=40, bins="log", cmap="viridis", mincnt=1)
ax[1].set_xlabel("MAF (likelihood EM)"); ax[1].set_ylabel("call rate"); ax[1].set_title("Call rate depends on allele frequency")
plt.colorbar(hb, ax=ax[1], label="SNPs (log)"); panel_label(ax[1], "B")

sub = snp1.sample(min(len(snp1), 30000), random_state=1)
ax[2].scatter(sub.call_rate_ctrl, sub.call_rate_case, s=3, alpha=.2, color=C["dark"], rasterized=True); ax[2].plot([0, 1], [0, 1], "r--", lw=.8)
ax[2].set_xlabel(f"call rate controls ({CONTROL_MORPH})"); ax[2].set_ylabel(f"call rate cases ({CASE_MORPH})"); ax[2].set_title("Missingness: cases vs controls"); panel_label(ax[2], "C")

ax[3].hist(-np.log10(np.maximum(snp1.p_diffmiss, 1e-300)), bins=50, color=C["fail"], alpha=.85)
ax[3].set_yscale("log"); ax[3].set_xlabel("−log10 P  (Fisher test of differential missingness)"); ax[3].set_ylabel("SNPs"); ax[3].set_title("Differential missingness")
panel_label(ax[3], "D")
plt.tight_layout(); savefig("snp_callrate"); plt.show()

print(f"median call rate {snp1.call_rate.median():.2f}, median data rate {snp1.data_rate.median():.2f}  ->  hard-calling discards {100*(1-snp1.call_rate.sum()/max(snp1.data_rate.sum(),1e-9)):.0f}% of the genotypes that had some data")
cr = pd.DataFrame({"threshold": [0.2, 0.5, 0.8, 0.9, 0.95, 0.99]})
cr["SNPs with call rate < t"] = [(snp1.call_rate < t).sum() for t in cr.threshold]; cr["% of SNPs"] = 100 * cr["SNPs with call rate < t"] / len(snp1)
cr["SNPs with data rate < t"] = [(snp1.data_rate < t).sum() for t in cr.threshold]
display(cr.round(2))

**Reading the plots**

* **A.** The call-rate distribution is shifted to the left of the data rate. At low coverage a large part of the SNPs will fall below the *standard* threshold of 0.95 — the 'standard' preset would remove almost the whole data set, which is why we also use an *adapted* threshold.
* **B.** Call rate **depends on allele frequency** (compare the colour pattern along the MAF axis). At low depth the posterior is driven by the prior: for a rare variant nearly everybody is homozygous for the common allele and one or two reads are enough to call it confidently, whereas for common variants the prior is flat and calls need more reads. This couples call rate and MAF (filtering on one affects the other).
* **C.** Points off the diagonal are SNPs with unequal missingness between groups. With a perfectly random missingness the cloud is symmetric around the diagonal.
* **D.** Under random missingness the p-values are uniform: the histogram of −log10 P decays smoothly. A tail of very small p-values is **differential missingness** (a typical cause of false positives).

## 3.2 Minor allele frequency

We compare the MAF from the likelihood EM (`maf`) with the MAF from hard calls (`maf_hard`) and look at the minor allele **count** (MAC), which is what determines whether a SNP is testable.

In [ ]:
# 3.2  MAF
fig, ax = plt.subplots(1, 4, figsize=(19, 4))
bins = np.linspace(0, 0.5, 51)
ax[0].hist(snp1.maf, bins=bins, color=C["case"], alpha=.85, label="likelihood EM")
ax[0].hist(snp1.maf_hard, bins=bins, histtype="step", color=C["fail"], lw=1.6, label="hard calls")
for nm, col in (("standard", C["fail"]), ("adapted", C["gold"])):
    ax[0].axvline(PRESETS[nm]["maf"], color=col, ls="--", label=f"{nm} MAF ≥ {PRESETS[nm]['maf']}")
ax[0].set_xlabel("minor allele frequency"); ax[0].set_ylabel("SNPs"); ax[0].set_title("Allele frequency spectrum"); ax[0].legend(fontsize=7); panel_label(ax[0], "A")

sub = snp1.sample(min(len(snp1), 30000), random_state=2)
ax[1].scatter(sub.maf, sub.maf_hard, s=3, alpha=.2, color=C["dark"], rasterized=True); ax[1].plot([0, .5], [0, .5], "r--", lw=.8)
ax[1].set_xlabel("MAF from likelihoods"); ax[1].set_ylabel("MAF from hard calls"); ax[1].set_title("Hard calls vs likelihoods"); panel_label(ax[1], "B")

ax[2].hist(snp1.mac.clip(upper=snp1.mac.quantile(.995)), bins=50, color=C["pass"], alpha=.85)
for nm, col in (("standard", C["fail"]), ("adapted", C["gold"])):
    if PRESETS[nm]["mac"] > 0: ax[2].axvline(PRESETS[nm]["mac"], color=col, ls="--", label=f"{nm}: MAC ≥ {PRESETS[nm]['mac']}")
ax[2].set_xlabel("minor allele count (2 × n_data × MAF)"); ax[2].set_ylabel("SNPs"); ax[2].set_title("Minor allele count"); ax[2].legend(fontsize=7); panel_label(ax[2], "C")

thr = np.array([0.01, 0.02, 0.05, 0.1, 0.2, 0.3])
ax[3].bar(range(len(thr)), [(snp1.maf < t).sum() for t in thr], color=C["grey"], label="likelihood EM")
ax[3].bar(np.arange(len(thr)) + .38, [(snp1.maf_hard < t).sum() for t in thr], width=.38, color=C["fail"], alpha=.8, label="hard calls")
ax[3].set_xticks(np.arange(len(thr)) + .19); ax[3].set_xticklabels([f"<{t}" for t in thr]); ax[3].set_ylabel("SNPs failing the MAF filter"); ax[3].set_xlabel("MAF threshold")
ax[3].set_title("SNPs removed by a MAF filter"); ax[3].legend(fontsize=7); panel_label(ax[3], "D")
plt.tight_layout(); savefig("snp_maf"); plt.show()
print(f"ANGSD called SNPs with -minMaf 0.05, so the smallest EM MAF in the file is {snp1.maf.min():.3f}; {(snp1.maf < 0.05).sum():,} SNPs ({100*(snp1.maf<0.05).mean():.1f}%) now have MAF < 0.05 (frequency re-estimated in this analysis).")
print(f"with n = {N} individuals, MAF = 0.05 corresponds to at most {2*N*0.05:.0f} copies of the minor allele")

**Reading the plots**

* **A, B.** Compare the hard-call and likelihood-based MAF. Hard-call MAF is computed from a *non-random subset* of the genotypes (those that were confident), so it deviates from the likelihood-based estimate; points away from the diagonal in B are SNPs where this selective calling matters. This is a reason to use the EM estimate for filtering.
* **C.** With few individuals, a MAF of 5% is just a handful of allele copies. We therefore combine MAF with a **minor allele count** criterion.
* **D.** How many SNPs would be removed by MAF thresholds at each level, using each definition.

## 3.3 Hardy–Weinberg equilibrium

We have four versions of the test:

| | Individuals | Test | Data |
|---|---|---|---|
| `lrt_p_all` | all | likelihood-ratio | **genotype likelihoods** |
| `lrt_p_ctrl` | controls only | likelihood-ratio | genotype likelihoods |
| `exact_p_all` | all | exact (Wigginton) | hard calls |
| `exact_p_ctrl` | controls only | exact (Wigginton) | hard calls |

### The de Finetti diagram
Each SNP is one point in the triangle: the corners are the genotypes AA, AB, BB; points on the **parabola** are in HWE. Points **below** the curve have a heterozygote **deficit**, points **above** have an **excess**. We draw it twice: from **hard calls** (what PLINK sees) and from the **likelihood-based** genotype frequency estimate (`f_AB = 2pq(1−F)`).

In [ ]:
# 3.3a  de Finetti diagrams
def triangle(ax, title):
    ax.plot([0, 1, .5, 0], [0, 0, np.sqrt(3) / 2, 0], color="k", lw=1)
    p = np.linspace(0, 1, 200); f1 = 2 * p * (1 - p); f2 = p * p
    ax.plot(f2 + f1 / 2, f1 * np.sqrt(3) / 2, color=C["fail"], lw=1.8, label="HWE parabola")
    ax.text(-.02, -.05, "AA", ha="right"); ax.text(1.02, -.05, "BB"); ax.text(.5, np.sqrt(3) / 2 + .03, "AB", ha="center")
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title)

def to_xy(f0, f1, f2): return f2 + f1 / 2, f1 * np.sqrt(3) / 2

c = S1["cnt"]["all"]; ok = (snp1.n_called >= max(0.5 * N, 5)).values
idx = np.where(ok)[0]; idx = rng.choice(idx, min(len(idx), 15000), replace=False) if len(idx) else idx
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
triangle(ax[0], "Hard calls (what PLINK sees)")
n = c[idx].sum(1); x, y = to_xy(c[idx, 0] / n, c[idx, 1] / n, c[idx, 2] / n)
ax[0].scatter(x, y, s=3, alpha=.2, color=C["dark"], rasterized=True); ax[0].legend(loc="upper left", fontsize=8)
p_, F_ = snp1.p.values[idx], snp1.F_all.values[idx]
f1 = 2 * p_ * (1 - p_) * (1 - F_); f2 = p_ * p_ + F_ * p_ * (1 - p_); f0 = 1 - f1 - f2
triangle(ax[1], "Likelihood-based estimate (all individuals)")
good = np.isfinite(f1)
sig = (snp1.lrt_p_all.values[idx] < 1e-4)
ax[1].scatter(*to_xy(f0[good & ~sig], f1[good & ~sig], f2[good & ~sig]), s=3, alpha=.2, color=C["dark"], rasterized=True)
ax[1].scatter(*to_xy(f0[good & sig], f1[good & sig], f2[good & sig]), s=5, alpha=.5, color=C["fail"], rasterized=True, label="HWE LRT P < 1e-4"); ax[1].legend(loc="upper left", fontsize=8)
ax[2].hist(snp1.het_hard_obs[ok], bins=50, alpha=.8, color=C["fail"], label="hard calls: observed heterozygote fraction", density=True)
exp_het = 2 * snp1.p * (1 - snp1.p)
ax[2].hist(exp_het[ok], bins=50, alpha=.6, color=C["pass"], label="expected under HWE (2pq)", density=True)
ax[2].set_xlabel("heterozygote frequency"); ax[2].set_title("Observed vs expected heterozygosity"); ax[2].legend(fontsize=8)
plt.tight_layout(); savefig("hwe_definetti"); plt.show()
print(f"mean observed hard-call heterozygote fraction {snp1.het_hard_obs[ok].mean():.3f} vs expected under HWE {exp_het[ok].mean():.3f} ")

**Reading the diagrams**

* **Left (hard calls).** Compare where the cloud lies relative to the parabola. Under random mating and complete data it is centred on the curve. If hard calls are selectively missing or miscalled the cloud shifts *below* (heterozygote deficit) or *above* (excess) the curve for **all** SNPs — a technical shift, not biology. The printout below the figure compares the observed and expected heterozygote fractions.
* **Middle (likelihoods).** The likelihood-based estimates use all reads, so the cloud should be centred much closer to the parabola. SNPs far above the parabola (strong heterozygote excess) are typical of **collapsed paralogs** (reads from two loci piling up on one reference site): red points are those with a significant HWE test.
* **Right.** The same information as histograms.

In [ ]:
# 3.3b  F vs MAF; histogram of F; Manhattan-style plot of HWE along the genome
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
sub = snp1.sample(min(len(snp1), 40000), random_state=3)
fail = sub.lrt_p_all < 1e-4
ax[0].scatter(sub.maf[~fail], sub.F_all[~fail], s=3, alpha=.2, color=C["grey"], rasterized=True)
ax[0].scatter(sub.maf[fail], sub.F_all[fail], s=5, alpha=.6, color=C["fail"], rasterized=True, label="LRT P < 1e-4")
ax[0].axhline(0, color="k", lw=.8); ax[0].set_xlabel("MAF"); ax[0].set_ylabel("inbreeding coefficient F (likelihood-based)"); ax[0].set_ylim(-1.05, 1.05)
ax[0].set_title("F vs MAF"); ax[0].legend(fontsize=8); panel_label(ax[0], "A")

ax[1].hist(snp1.F_all.dropna(), bins=np.linspace(-1, 1, 81), color=C["case"], alpha=.85, label="likelihood-based F")
ax[1].hist(snp1.F_hard.dropna(), bins=np.linspace(-1, 1, 81), histtype="step", color=C["fail"], lw=1.6, label="hard-call F")
ax[1].axvline(0, color="k", lw=.8); ax[1].set_xlabel("F  (<0: heterozygote excess, >0: deficit)"); ax[1].set_ylabel("SNPs"); ax[1].set_title("Distribution of F")
ax[1].legend(fontsize=8); panel_label(ax[1], "B")

# genome-wide plot
contigs = list(dict.fromkeys(snp1.contig)); off = {}; cum = 0; ticks = []
for ct in contigs:
    mx = snp1.pos[snp1.contig == ct].max(); off[ct] = cum; ticks.append(cum + mx / 2); cum += mx + 1e5
x = snp1.pos.values + snp1.contig.map(off).values; y = -np.log10(np.maximum(snp1.lrt_p_all.values, 1e-300))
colr = np.where(snp1.contig.map({c_: i % 2 for i, c_ in enumerate(contigs)}).values == 0, C["case"], C["dark"])
sub = np.where(y > 2)[0]; sub = np.union1d(sub, rng.choice(len(y), min(len(y), 30000), replace=False))
ax[2].scatter(x[sub], y[sub], s=3, c=colr[sub], alpha=.5, rasterized=True)
ax[2].axhline(-np.log10(1e-4), color=C["gold"], ls="--"); ax[2].axhline(6, color=C["fail"], ls="--")
ax[2].set_xlabel("position along the (concatenated) chromosomes"); ax[2].set_ylabel("−log10 P  (HWE likelihood-ratio test)"); ax[2].set_title("HWE failures along the genome")
ax[2].set_xticks([]); panel_label(ax[2], "C")
plt.tight_layout(); savefig("hwe_F_genome"); plt.show()
print(f"SNPs with heterozygote EXCESS  (F < −0.3): {(snp1.F_all < -0.3).sum():,}  ({100*(snp1.F_all < -0.3).mean():.1f}%)   -> candidate collapsed paralogs")
print(f"SNPs with heterozygote DEFICIT (F > +0.3): {(snp1.F_all > 0.3).sum():,}  ({100*(snp1.F_all > 0.3).mean():.1f}%)")

**Reading the plots**

* **A.** For a well-behaved SNP F ≈ 0 for every MAF. A **cloud with strongly negative F** (heterozygote excess) is the paralog signature; a cloud with positive F is a deficit (structure/inbreeding/genotyping errors). Rare SNPs have a widely scattered F simply because of sampling noise (few heterozygotes expected).
* **B.** Compare the hard-call F with the likelihood-based F: a shift of the whole distribution away from 0 is the selective-calling bias; the likelihood-based F should be centred closer to 0.
* **C.** HWE failures often **cluster** in specific genomic regions (repeats, regions with copy-number variation, inversions) rather than being spread uniformly; a few contigs with many failures are a hint of poorly assembled or repetitive regions.

### QQ plots: how many HWE failures are expected by chance?

Under the null hypothesis, p-values are uniform: the −log10 P QQ plot follows the diagonal. Deviation at the **top-right** (very small p-values) = true departures (errors, paralogs, structure); deviation across the **whole** curve = test is mis-calibrated (e.g. because the sample is structured, or hard-call bias).

In [ ]:
# 3.3c  QQ plots for the four versions of the HWE test
def qq(ax, p, label, color):
    p = np.sort(p[np.isfinite(p)]); n = len(p)
    if n == 0: return
    k = np.unique(np.round(np.geomspace(1, n, 2500)).astype(int)) - 1
    ax.plot(-np.log10((k + 0.5) / n), -np.log10(np.maximum(p[k], 1e-300)), ".", ms=3, color=color, label=f"{label} (n={n:,})")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
tests = [("lrt_p_all", "GL-LRT, all", C["case"]), ("lrt_p_ctrl", f"GL-LRT, controls", C["pass"]),
         ("exact_p_all", "exact, hard calls, all", C["fail"]), ("exact_p_ctrl", "exact, hard calls, controls", C["gold"])]
for a in ax[:1]:
    for col, lab, cl in tests: qq(a, snp1[col].values, lab, cl)
    mx = a.get_xlim()[1]; a.plot([0, mx], [0, mx], "k--", lw=.8); a.set_xlabel("expected −log10 P"); a.set_ylabel("observed −log10 P"); a.set_title("QQ plots of HWE tests"); a.legend(fontsize=7)
panel_label(ax[0], "A")
sub = snp1.sample(min(len(snp1), 30000), random_state=4)
a = ax[1]; a.scatter(-np.log10(np.maximum(sub.exact_p_all, 1e-300)), -np.log10(np.maximum(sub.lrt_p_all, 1e-300)), s=3, alpha=.25, color=C["dark"], rasterized=True)
a.set_xlabel("−log10 P exact test (hard calls)"); a.set_ylabel("−log10 P likelihood-ratio test"); a.set_title("Two ways to test the same SNP"); panel_label(a, "B")
# direction of failures
a = ax[2]
rows = []
for col, lab, _ in tests:
    f = snp1[col] < 1e-4
    rows.append((lab, int((f & (snp1.F_all < 0)).sum()), int((f & (snp1.F_all >= 0)).sum())))
t = pd.DataFrame(rows, columns=["test", "het. excess (F<0)", "het. deficit (F≥0)"])
t.set_index("test").plot.barh(stacked=True, ax=a, color=[C["gold"], C["fail"]]); a.set_xlabel("SNPs with P < 1e-4"); a.set_ylabel(""); a.set_title("Direction of the HWE failures"); panel_label(a, "C")
plt.tight_layout(); savefig("hwe_qq"); plt.show()

hw = pd.DataFrame({"P threshold": [0.05, 1e-3, 1e-4, 1e-6]})
for col, lab, _ in tests: hw[lab] = [int((snp1[col] < t).sum()) for t in hw["P threshold"]]
display(hw)

**Reading the table and plots**

* **The two kinds of test usually disagree** — as the simulation predicted. Compare the number of failures: the hard-call exact test is affected by selective calling (panel C shows the direction of its failures), while the likelihood test is not and can flag both directions.
* The **controls-only** tests have **less power** (half the individuals), so fewer SNPs fail.
* A QQ curve which stays on the diagonal for most of the range and bends up only at the end is what we want to see for the likelihood-based test.

### Power of the HWE test as a function of sample size

How many individuals do we need? We simulate genotypes (not reads) for a SNP with MAF = 0.3 and inbreeding coefficients F = −0.3, −0.15, +0.2, +0.5 (at MAF 0.3 the heterozygote excess cannot go below F = −0.43), run the exact test and count how often P is below a given α. This is the *best case*: perfect genotypes.

In [ ]:
# 3.3d  Power of the exact HWE test
def sim_power(n, F, p=0.3, reps=1000, alphas=(1e-2, 1e-4, 1e-6), seed=0):
    r = np.random.default_rng(seed)
    f = np.array([(1 - p)**2 + F * p * (1 - p), 2 * p * (1 - p) * (1 - F), p**2 + F * p * (1 - p)])
    c = r.multinomial(n, f, size=reps)
    P = exact_p_for_counts(c[:, 0], c[:, 1], c[:, 2])
    return [np.mean(P < a) for a in alphas]
ns = [10, 20, 50, 100, 200, 400, 1000, 3000]
alphas = (1e-2, 1e-4, 1e-6)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for F, ls in ((-0.3, "-"), (-0.15, "--"), (0.2, ":"), (0.5, "-.")):
    pw = np.array([sim_power(n, F, alphas=alphas) for n in ns])
    for j, (a_, col) in enumerate(zip(alphas, [C["pass"], C["gold"], C["fail"]])):
        if j in (1, 2): ax[j - 1].plot(ns, pw[:, j], ls=ls, color=col, marker="o", ms=3, label=f"F = {F:+.1f}")
for j, a_ in enumerate(alphas[1:]):
    ax[j].set_xscale("log"); ax[j].set_ylim(-.02, 1.02); ax[j].axvline(N, color="k", lw=.8); ax[j].text(N * 1.05, .5, f"this data set\n(n={N})", fontsize=8)
    ax[j].set_xlabel("number of individuals"); ax[j].set_ylabel("power"); ax[j].set_title(f"Power of the exact HWE test at P < {a_:g}  (MAF 0.3)"); ax[j].legend(fontsize=8)
plt.tight_layout(); savefig("hwe_power"); plt.show()
print("with", N, "individuals, power at P<1e-6 for F = −0.3 and F = +0.5:", [round(x, 3) for x in (sim_power(N, -0.3, alphas=(1e-6,))[0], sim_power(N, 0.5, alphas=(1e-6,))[0])])

**Take-home.** With a few tens of individuals the exact test has **practically no power** at the strict thresholds used in GWAS (10⁻⁶): HWE filtering can only catch the most extreme departures. With hundreds of individuals it becomes informative; with thousands it can detect small departures. This is why HWE thresholds should depend on the sample size, and why the *adapted* preset uses a less strict P-value computed from *all* individuals.

## 3.4 Combining the criteria — which SNPs are problematic?

We now apply both threshold sets (`standard`, `adapted`) to the same table of SNP statistics, and look at:

1. how many SNPs fail **each** criterion on its own,
2. how the failures **overlap** (UpSet plots: each bar is a group of SNPs failing exactly that combination),
3. how many SNPs **remain** when the filters are applied **in sequence** (call rate → MAF/MAC → HWE → differential missingness).

In [ ]:
# 3.4  Variant flags for a preset
def variant_flags(T, ps):
    f = pd.DataFrame(index=T.index)
    f["low call rate"] = T.call_rate < ps["snp_call"]
    f["low MAF/MAC"] = (T.maf < ps["maf"]) | (T.mac < ps["mac"])
    g = "ctrl" if ps["hwe_group"] == "controls" else "all"
    col = ("lrt_p_" if ps["hwe_test"] == "lrt_gl" else "exact_p_") + g
    f["HWE fail"] = (T[col] < ps["hwe_p"]).fillna(False)
    f["differential missingness"] = (T.p_diffmiss < ps["diffmiss_p"]).fillna(False)
    return f

def upset(flags, title, fig=None, max_bars=12, color=None):
    names = list(flags.columns); k = len(names)
    key = (flags.values.astype(int) * (2 ** np.arange(k))).sum(1)
    cnt = np.bincount(key, minlength=2 ** k)
    n_pass = cnt[0]
    combos = [(c, cnt[c]) for c in np.argsort(-cnt) if c != 0 and cnt[c] > 0][:max_bars]
    totals = flags.sum().values
    fig = fig or plt.figure(figsize=(8, 4.2))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.2, 4], height_ratios=[2.5, 1.4], hspace=0.03, wspace=0.03)
    ax_bar = fig.add_subplot(gs[0, 1]); ax_mat = fig.add_subplot(gs[1, 1], sharex=ax_bar); ax_tot = fig.add_subplot(gs[1, 0])
    xs = np.arange(len(combos))
    ax_bar.bar(xs, [c[1] for c in combos], color=color or C["dark"])
    for x_, c_ in zip(xs, combos): ax_bar.text(x_, c_[1], f"{c_[1]:,}", ha="center", va="bottom", fontsize=7)
    ax_bar.set_ylabel("variants"); ax_bar.set_title(f"{title}\n(pass all: {n_pass:,} of {len(flags):,})", fontsize=10); plt.setp(ax_bar.get_xticklabels(), visible=False)
    for j in range(k):
        ax_mat.axhline(j, color="#dee2e6", lw=8, zorder=0)
    for x_, (c_, _) in zip(xs, combos):
        on = [j for j in range(k) if (c_ >> j) & 1]
        ax_mat.scatter([x_] * len(on), on, color=C["dark"], s=40, zorder=3)
        if len(on) > 1: ax_mat.plot([x_, x_], [min(on), max(on)], color=C["dark"], lw=2, zorder=2)
    ax_mat.set_yticks([]); ax_mat.set_xticks([]); ax_mat.set_ylim(-.6, k - .4); ax_mat.set_xlim(-.7, len(combos) - .3)
    for s in ("left", "bottom"): ax_mat.spines[s].set_visible(False)
    ax_tot.barh(range(k), totals, color=C["grey"]); ax_tot.invert_xaxis(); ax_tot.set_yticks(range(k)); ax_tot.set_yticklabels(names, fontsize=8); ax_tot.yaxis.tick_right()
    ax_tot.set_ylim(-.6, k - .4); ax_tot.set_xlabel("total failing", fontsize=8); ax_tot.spines["left"].set_visible(False)
    for j, t_ in enumerate(totals): ax_tot.text(t_, j, f"{t_:,} ", ha="right", va="center", fontsize=7)
    return fig

FLAGS1 = {nm: variant_flags(snp1, ps) for nm, ps in PRESETS.items()}
fig = plt.figure(figsize=(16, 5))
for i, (nm, fl) in enumerate(FLAGS1.items()):
    sub = fig.add_subfigure(plt.GridSpec(1, 2)[i])
    upset(fl, f"{nm} thresholds", fig=sub, color=C["fail"] if nm == "standard" else C["gold"])
savefig("variant_upset"); plt.show()

tab = pd.DataFrame({nm: fl.sum() for nm, fl in FLAGS1.items()}); tab.loc["fail ≥ 1 criterion"] = [int(fl.any(axis=1).sum()) for fl in FLAGS1.values()]
tab.loc["PASS all"] = [int((~fl.any(axis=1)).sum()) for fl in FLAGS1.values()]
tab.loc["% of SNPs passing"] = [100 * (~fl.any(axis=1)).mean() for fl in FLAGS1.values()]
display(tab.round(1))

**Reading the UpSet plots.** Each column of dots is a *combination* of failed criteria, the bar above gives the number of SNPs failing *exactly* that combination (not the union). Bars with a single dot are SNPs that fail only one criterion. The horizontal bars on the left give the total number failing each criterion. Overlaps tell you how redundant the filters are: e.g. most SNPs with a low call rate also have a low MAF (rare variants are called less well), so removing one set removes the other.

In [ ]:
# 3.5  Sequential funnel: how many variants remain after each filter?
ORDER = ["low call rate", "low MAF/MAC", "HWE fail", "differential missingness"]
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1.3, 1]})
funnel = {}
for nm, fl in FLAGS1.items():
    keep = np.ones(len(fl), bool); rows = [("start (ANGSD SNPs)", keep.sum())]
    for o in ORDER: keep &= ~fl[o].values; rows.append((f"+ {o}", keep.sum()))
    funnel[nm] = rows
f_df = pd.DataFrame({nm: [r[1] for r in rows] for nm, rows in funnel.items()}, index=[r[0] for r in funnel["standard"]])
y = np.arange(len(f_df)); h = 0.38
ax[0].barh(y - h / 2, f_df["standard"], height=h, color=C["fail"], label="standard"); ax[0].barh(y + h / 2, f_df["adapted"], height=h, color=C["gold"], label="adapted")
for yy, v in zip(y - h / 2, f_df["standard"]): ax[0].text(v, yy, f" {v:,}", va="center", fontsize=8)
for yy, v in zip(y + h / 2, f_df["adapted"]): ax[0].text(v, yy, f" {v:,}", va="center", fontsize=8)
ax[0].set_yticks(y); ax[0].set_yticklabels(f_df.index); ax[0].invert_yaxis(); ax[0].set_xlabel("SNPs remaining"); ax[0].set_title("Sequential filtering"); ax[0].legend(); panel_label(ax[0], "A")
# each filter alone
alone = pd.DataFrame({nm: [len(fl) - int(fl[o].sum()) for o in ORDER] for nm, fl in FLAGS1.items()}, index=ORDER)
alone.plot.barh(ax=ax[1], color=[C["fail"], C["gold"]]); ax[1].invert_yaxis(); ax[1].set_xlabel("SNPs remaining if only this filter is applied"); ax[1].set_title("Each filter alone"); panel_label(ax[1], "B")
for cont in ax[1].containers: ax[1].bar_label(cont, fmt="{:,.0f}", fontsize=7, padding=2)
plt.tight_layout(); savefig("variant_funnel"); plt.show()
display(f_df.assign(**{"standard %": (100 * f_df.standard / f_df.standard.iloc[0]).round(1), "adapted %": (100 * f_df.adapted / f_df.adapted.iloc[0]).round(1)}))

## 3.6 How sensitive are the numbers to the thresholds?

QC thresholds are **choices**; a sound analysis shows how much the data set depends on them. The curves below show how many SNPs survive over a range of thresholds for each criterion (vertical lines: the two presets).

In [ ]:
# 3.6  Threshold sweeps
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
t = np.linspace(0, 1, 101)
ax[0].plot(t, [(snp1.call_rate >= x).sum() for x in t], color=C["case"], lw=2, label="call rate (hard calls)")
ax[0].plot(t, [(snp1.data_rate >= x).sum() for x in t], color=C["grey"], lw=2, label="data rate (any reads)")
ax[0].set_xlabel("minimum call rate"); ax[0].set_ylabel("SNPs retained"); ax[0].set_title("SNP call rate"); ax[0].legend(fontsize=8)
tm = np.linspace(0, 0.5, 101)
ax[1].plot(tm, [(snp1.maf >= x).sum() for x in tm], color=C["case"], lw=2, label="MAF (likelihood)"); ax[1].plot(tm, [(snp1.maf_hard >= x).sum() for x in tm], color=C["fail"], lw=2, label="MAF (hard calls)")
ax[1].set_xlabel("minimum MAF"); ax[1].set_title("MAF"); ax[1].legend(fontsize=8)
ms = np.arange(0, 41)
ax[2].plot(ms, [(snp1.mac >= x).sum() for x in ms], color=C["pass"], lw=2); ax[2].set_xlabel("minimum minor allele count"); ax[2].set_title("MAC")
th = np.logspace(-12, 0, 100)
for col, lab, cl in tests: ax[3].plot(th, [(snp1[col].fillna(1) >= x).sum() for x in th], color=cl, lw=2, label=lab)
ax[3].set_xscale("log"); ax[3].set_xlabel("HWE P-value threshold (remove SNPs with P < threshold)"); ax[3].set_title("HWE"); ax[3].legend(fontsize=7)
for a in ax: a.set_ylim(0, len(snp1) * 1.02)
for nm, col in (("standard", C["fail"]), ("adapted", C["gold"])):
    ps = PRESETS[nm]; ax[0].axvline(ps["snp_call"], color=col, ls="--"); ax[1].axvline(ps["maf"], color=col, ls="--")
    if ps["mac"]: ax[2].axvline(ps["mac"], color=col, ls="--")
    ax[3].axvline(ps["hwe_p"], color=col, ls="--")
plt.tight_layout(); savefig("variant_sweeps"); plt.show()

# Part 4 — Sample QC (and the final variant filtering)

Following the order of operations (0.6):

1. **Lenient variant filter** — keep SNPs with call rate ≥ `LENIENT_SNP_CALL`: they carry enough information for the sample statistics;
2. **Sample QC** with those SNPs: call rate, heterozygosity, relatedness (GRM), PCA outliers;
3. **Remove** the flagged individuals according to the `ACTIVE` preset;
4. **Re-scan** the BEAGLE file with the retained individuals (allele frequencies, call rates and HWE change once individuals are removed), and apply the **strict variant filters**.

## 4.1 Per-sample metrics

| Metric | Definition |
|---|---|
| **data rate** | fraction of the SNPs with reads in this individual (a proxy for sequencing depth) |
| **call rate** | fraction of the SNPs with a confident hard call |
| **heterozygosity** | fraction of heterozygous genotypes among the hard calls |
| **F** (PLINK `--het`) | `(O_hom − E_hom)/(N − E_hom)`; E_hom = expected number of homozygous genotypes from the allele frequencies. F > 0 = homozygote excess |

In [ ]:
# 4.1  Per-sample metrics from hard calls, on the SNPs passing the lenient filter
def sample_metrics(S, T, snp_mask):
    G, V, p = S["G"][snp_mask], S["valid"][snp_mask], T.p.values[snp_mask]
    called = G >= 0; m = int(snp_mask.sum())
    n_called, n_data = called.sum(0), V.sum(0)
    het, hom = (G == 1).sum(0), ((G == 0) | (G == 2)).sum(0)
    E_hom = ((1 - 2 * p * (1 - p))[:, None] * called).sum(0)
    out = pd.DataFrame({"sampleID": [SAMPLES[i] for i in S["keep_idx"]], "morph": pheno.morph.values[S["keep_idx"]]})
    out["call_rate"] = n_called / m; out["data_rate"] = n_data / m
    out["het_rate"] = het / np.maximum(n_called, 1)
    out["F_het"] = (hom - E_hom) / np.maximum(n_called - E_hom, 1e-9)
    if "mean_depth" in pheno.columns: out["mean_depth"] = pheno.mean_depth.values[S["keep_idx"]]
    return out

lenient = (snp1.call_rate >= LENIENT_SNP_CALL).values
print(f"lenient SNP filter (call rate ≥ {LENIENT_SNP_CALL}): {lenient.sum():,} of {len(snp1):,} SNPs retained for the sample QC")
sm1 = sample_metrics(S1, snp1, lenient)
display(sm1.round(3).head(10))

In [ ]:
# 4.2  Sample call rate, data rate, heterozygosity
fig, ax = plt.subplots(1, 4, figsize=(20, 4.6))
order = np.argsort(-sm1.data_rate.values); colr = np.where(sm1.morph == CASE_MORPH, C["case"], C["ctrl"])
x = np.arange(len(sm1))
ax[0].bar(x - .2, sm1.data_rate.values[order], width=.4, color=C["grey"], label="data rate"); ax[0].bar(x + .2, sm1.call_rate.values[order], width=.4, color=colr[order], label="call rate")
ax[0].set_xticks(x); ax[0].set_xticklabels(sm1.sampleID.values[order], rotation=90, fontsize=7); ax[0].set_ylabel("fraction of SNPs"); ax[0].set_title("Sample data rate and call rate")
ax[0].legend(handles=[Patch(color=C["grey"], label="data rate"), Patch(color=C["case"], label=f"call rate, {CASE_MORPH}"), Patch(color=C["ctrl"], label=f"call rate, {CONTROL_MORPH}")], fontsize=7)
panel_label(ax[0], "A")

ax[1].scatter(sm1.data_rate, sm1.het_rate, c=colr, s=45, edgecolor="white")
b, a_ = np.polyfit(sm1.data_rate, sm1.het_rate, 1); xx = np.linspace(sm1.data_rate.min(), sm1.data_rate.max(), 10)
ax[1].plot(xx, a_ + b * xx, color=C["dark"], lw=1, ls="--"); r_ = np.corrcoef(sm1.data_rate, sm1.het_rate)[0, 1]
ax[1].set_xlabel("data rate (≈ depth)"); ax[1].set_ylabel("heterozygosity (hard calls)"); ax[1].set_title(f"Heterozygosity tracks depth (r = {r_:.2f})"); panel_label(ax[1], "B")

ax[2].hist(sm1.F_het, bins=15, color=C["case"], alpha=.85); ax[2].axvline(sm1.F_het.mean(), color="k");
for k in (-3, 3): ax[2].axvline(sm1.F_het.mean() + k * sm1.F_het.std(), color=C["fail"], ls="--")
ax[2].set_xlabel("F (PLINK --het)"); ax[2].set_ylabel("individuals"); ax[2].set_title("Sample inbreeding coefficient, ±3 SD"); panel_label(ax[2], "C")

ax[3].scatter(sm1.call_rate, sm1.F_het, c=colr, s=45, edgecolor="white"); ax[3].set_xlabel("call rate"); ax[3].set_ylabel("F (hard calls)"); ax[3].set_title("Heterozygote deficit vs call rate"); panel_label(ax[3], "D")
plt.tight_layout(); savefig("sample_metrics"); plt.show()
print(f"correlation of heterozygosity with data rate: {r_:.2f}  (if it is strongly positive or negative, heterozygosity reflects depth, a technical effect, and not biology)")

**Reading the plots**

* **A.** Grey = share of SNPs with reads; coloured = share with a confident hard call. Coverage differs a lot between individuals; individuals with a very low data rate contribute little.
* **B.** *The key plot for low-coverage QC.* The heterozygosity of the hard calls depends on depth, because the genotypes that become confident hard calls change with the number of reads (the sign of the relationship depends on cutoff and allele frequencies; r is printed in the title). A raw "±3 SD of heterozygosity" rule would then flag the **least** (or most) covered individuals as inbred/contaminated, which is wrong. The *adapted* rule therefore looks at the **residual** from the heterozygosity–depth regression: an individual with *more* heterozygotes than expected for its depth is a candidate for contamination or paralog-rich mapping; *fewer* than expected may be real inbreeding or allele dropout.
* **C, D.** Distribution of F and its relation to call rate.

## 4.3 Relatedness: the genomic relationship matrix (GRM)

For individuals *j* and *k* and SNPs *s* with allele frequency *p_s* and dosage *x* (expected number of B alleles):

$$ \text{GRM}_{jk} \;=\; \frac{1}{m}\sum_{s} \frac{(x_{js}-2p_s)(x_{ks}-2p_s)}{2p_s(1-p_s)} $$

For unrelated individuals GRM ≈ 0, the diagonal ≈ 1 (+F). The kinship coefficient is **φ = GRM / 2**. We use the **thinned** SNP set (one SNP per `THIN_KB` kb with MAF ≥ 5% and good coverage) so that SNPs in strong linkage disequilibrium do not dominate, and the **posterior dosage** (not the hard calls) to keep uncertainty. Caveat: a poorly covered individual's dosage is pulled towards its prior (2p), which *shrinks* its relatedness to everybody towards 0: low-coverage duplicates are therefore under-estimated.

In [ ]:
# 4.3  GRM and PCA from the thinned SNP set
def grm_and_pca(S, T, snp_mask_full=None):
    rows = S["thin_rows"]; DS = S["DS"].astype(np.float64)
    if snp_mask_full is not None:
        sel = snp_mask_full[rows]; rows, DS = rows[sel], DS[sel]
    p = T.p.values[rows]; ok = (np.minimum(p, 1 - p) > 0.02)
    rows, DS, p = rows[ok], DS[ok], p[ok]
    if len(rows) > MAX_PCA_SNPS:
        k = np.sort(rng.choice(len(rows), MAX_PCA_SNPS, replace=False)); rows, DS, p = rows[k], DS[k], p[k]
    Z = (DS - 2 * p[:, None]) / np.sqrt(2 * p * (1 - p))[:, None]
    GRM = Z.T @ Z / len(rows)
    w, v = np.linalg.eigh(GRM); o = np.argsort(w)[::-1]; w, v = w[o], v[:, o]
    return GRM, w, v, len(rows)

GRM1, w1, v1, m_grm1 = grm_and_pca(S1, snp1, lenient)
NPC_SHOW = min(5, N - 1)
pcs1 = pd.DataFrame(v1[:, :NPC_SHOW] * np.sqrt(np.maximum(w1[:NPC_SHOW], 0)), columns=[f"PC{i+1}" for i in range(NPC_SHOW)])
pcs1.insert(0, "sampleID", sm1.sampleID.values)
print(f"{m_grm1:,} thinned SNPs used for the GRM; variance explained by PC1–PC{NPC_SHOW}: {np.round(100*w1[:NPC_SHOW]/np.sum(np.maximum(w1,0)),1)} %")

kin = GRM1 / 2; np.fill_diagonal(kin, np.nan)
pairs = pd.DataFrame([(sm1.sampleID[i], sm1.sampleID[j], kin[i, j], i, j) for i in range(N) for j in range(i + 1, N)], columns=["id1", "id2", "kinship", "i", "j"]).sort_values("kinship", ascending=False)
fig, ax = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw={"width_ratios": [1.15, 1, 1]})
o = np.lexsort((sm1.sampleID.values, sm1.morph.values)); M_ = GRM1[np.ix_(o, o)]
im = ax[0].imshow(M_ / 2, cmap="RdBu_r", vmin=-0.25, vmax=0.25); ax[0].grid(False)
ax[0].set_xticks(range(N)); ax[0].set_xticklabels([f"{sm1.sampleID[i]}" for i in o], rotation=90, fontsize=6); ax[0].set_yticks(range(N)); ax[0].set_yticklabels([f"{sm1.sampleID[i]} ({sm1.morph[i]})" for i in o], fontsize=6)
plt.colorbar(im, ax=ax[0], shrink=.8, label="kinship φ (GRM/2; diagonal shown too)"); ax[0].set_title("Genomic relationship matrix"); panel_label(ax[0], "A")
ax[1].hist(pairs.kinship, bins=40, color=C["grey"]);
for t_, lab in ((0.354, "duplicate/MZ"), (0.177, "1st degree"), (0.0884, "2nd degree")): ax[1].axvline(t_, color=C["fail"], ls="--"); ax[1].text(t_, ax[1].get_ylim()[1] * .95, lab, rotation=90, va="top", ha="right", fontsize=7)
ax[1].set_xlabel("pairwise kinship φ"); ax[1].set_ylabel("pairs"); ax[1].set_title(f"{len(pairs):,} pairs of individuals"); panel_label(ax[1], "B")
ax[2].axis("off"); top = pairs.head(10).copy(); top["kinship"] = top.kinship.round(3)
ax[2].table(cellText=top[["id1", "id2", "kinship"]].values, colLabels=["individual 1", "individual 2", "φ"], loc="center", cellLoc="center").scale(1, 1.3)
ax[2].set_title("Most related pairs", pad=14); panel_label(ax[2], "C")
plt.tight_layout(); savefig("sample_relatedness"); plt.show()
print(f"pairs with φ > 0.354 (duplicate/MZ): {(pairs.kinship > 0.354).sum()}  | φ > 0.177 (first-degree): {(pairs.kinship > 0.177).sum()}  | φ > 0.088 (second-degree): {(pairs.kinship > 0.0884).sum()}")

**Reading the plots.** In a sample of unrelated individuals all off-diagonal cells are close to 0 (white). Red off-diagonal blocks or isolated red cells are related individuals or population sub-structure (blocks of individuals of the same population are mildly positive relative to the rest). The histogram shows where the pairwise values fall relative to the conventional degrees of relationship. **Remember** that low-coverage individuals have their relatedness shrunk towards zero.

## 4.4 PCA outliers and structure

The principal components are the eigenvectors of the GRM. We look at the first PCs for (a) **outliers** (far from the cloud), (b) **structure** (clusters), (c) **technical effects** (do the PCs correlate with data rate or with the phenotype?).

In [ ]:
# 4.4  PCA
def pc_z(pcs, robust):
    Z = {}
    for c in [c for c in pcs.columns if c.startswith("PC")]:
        Z[c] = mad_z(pcs[c]) if robust else (pcs[c] - pcs[c].mean()) / pcs[c].std()
    return pd.DataFrame(Z)

zr = pc_z(pcs1, True)
fig, ax = plt.subplots(1, 4, figsize=(20, 4.6))
ev = 100 * np.maximum(w1, 0) / np.maximum(w1, 0).sum()
ax[0].bar(range(1, min(10, N) + 1), ev[:min(10, N)], color=C["grey"]); ax[0].set_xlabel("PC"); ax[0].set_ylabel("% variance"); ax[0].set_title("Scree plot"); panel_label(ax[0], "A")
for a_, (i, j), lab in zip(ax[1:3], [(1, 2), (3, 4)], "BC"):
    if f"PC{j}" not in pcs1: a_.axis("off"); continue
    for mo, cl in ((CASE_MORPH, C["case"]), (CONTROL_MORPH, C["ctrl"])):
        s_ = (sm1.morph == mo).values
        a_.scatter(pcs1.loc[s_, f"PC{i}"], pcs1.loc[s_, f"PC{j}"], s=30 + 250 * sm1.data_rate[s_], color=cl, alpha=.8, edgecolor="white", label=f"{mo} (n={s_.sum()})")
    out_ = (zr[[f"PC{i}", f"PC{j}"]].abs() > 4).any(axis=1).values
    for k in np.where(out_)[0]: a_.annotate(sm1.sampleID[k], (pcs1[f"PC{i}"][k], pcs1[f"PC{j}"][k]), fontsize=7)
    a_.set_xlabel(f"PC{i} ({ev[i-1]:.1f}%)"); a_.set_ylabel(f"PC{j} ({ev[j-1]:.1f}%)"); a_.set_title(f"PC{i} vs PC{j} (size = data rate)"); a_.legend(fontsize=8); panel_label(a_, lab)
ax[3].scatter(sm1.data_rate, pcs1.PC1, c=colr, s=45, edgecolor="white"); ax[3].set_xlabel("data rate"); ax[3].set_ylabel("PC1"); r2 = np.corrcoef(sm1.data_rate, pcs1.PC1)[0, 1]
ax[3].set_title(f"Is PC1 just depth? (r = {r2:.2f})"); panel_label(ax[3], "D")
plt.tight_layout(); savefig("sample_pca"); plt.show()
for c in pcs1.columns[1:]:
    y = (sm1.morph == CASE_MORPH).astype(float).values; rr, pv = stats.pointbiserialr(y, pcs1[c])
    print(f"{c}: association with morph  r = {rr:+.2f} (P = {pv:.3f});   with data rate r = {np.corrcoef(sm1.data_rate, pcs1[c])[0,1]:+.2f}")
print(f"\nThe maximum possible z-score in a sample of {N} individuals is {(N-1)/np.sqrt(N):.2f}: the classical '6 SD' EIGENSOFT rule cannot flag anyone in a sample this small; we use a robust (median/MAD) z-score for the 'adapted' preset.")

## 4.5 Combining everything: which individuals are problematic?

For each preset we flag individuals by:

| Flag | `standard` | `adapted` |
|---|---|---|
| **low call rate** | call rate < 0.95 | call rate < median − 3 MAD *(automatic, robust)* |
| **heterozygosity** | \|z(F)\| > 3 (SD) | \|z\| > 3 of the **depth-adjusted** heterozygosity residual (robust) |
| **related** | φ > 0.177 (first-degree or closer); the lower-call-rate member is flagged | φ > 0.354 (duplicates only) |
| **PCA outlier** | \|z\| > 6 SD on any of PC1–PC5 | \|z\| > 6 *robust* on any of PC1–PC5 |

In [ ]:
# 4.5  Sample flags for both presets
def sample_flags(sm, pcs, GRM, ps):
    f = pd.DataFrame(index=sm.index)
    cr_thr = ps["sample_call"] if ps["sample_call"] is not None else sm.call_rate.median() - SAMPLE_CALL_AUTO_MADS * 1.4826 * np.median(np.abs(sm.call_rate - sm.call_rate.median()))
    f["low call rate"] = sm.call_rate < cr_thr
    if ps["het_adjust"]:
        b, a_ = np.polyfit(sm.data_rate, sm.het_rate, 1); res = sm.het_rate - (a_ + b * sm.data_rate); z = mad_z(res)
    else:
        z = (sm.F_het - sm.F_het.mean()) / sm.F_het.std()
    f["heterozygosity outlier"] = np.abs(z) > ps["het_sd"]
    kin = GRM / 2; np.fill_diagonal(kin, 0); rel = np.zeros(len(sm), bool); kept = np.ones(len(sm), bool)
    ii, jj = np.where(np.triu(kin) > ps["kin"])
    for i, j in sorted(zip(ii, jj), key=lambda t: -kin[t]):
        if kept[i] and kept[j]:
            lose = i if sm.call_rate[i] <= sm.call_rate[j] else j; kept[lose] = False; rel[lose] = True
    f["related (lower-quality member)"] = rel
    zp = pc_z(pcs, ps["pca_robust"]); f["PCA outlier"] = (zp.abs() > ps["pca_sd"]).any(axis=1).values
    return f, cr_thr

SFLAGS1, SCR = {}, {}
for nm, ps in PRESETS.items(): SFLAGS1[nm], SCR[nm] = sample_flags(sm1, pcs1, GRM1, ps)
fig = plt.figure(figsize=(16, 4.6))
for i, (nm, fl) in enumerate(SFLAGS1.items()):
    sub = fig.add_subfigure(plt.GridSpec(1, 2)[i]); upset(fl, f"{nm} thresholds: flagged individuals", fig=sub, color=C["fail"] if nm == "standard" else C["gold"])
savefig("sample_upset"); plt.show()
tabs = pd.DataFrame({nm: fl.sum() for nm, fl in SFLAGS1.items()}); tabs.loc["flagged ≥ 1 criterion"] = [int(fl.any(axis=1).sum()) for fl in SFLAGS1.values()]
tabs.loc["retained"] = [int((~fl.any(axis=1)).sum()) for fl in SFLAGS1.values()]
display(tabs)
print(f"sample call-rate threshold: standard {SCR['standard']:.2f}, adapted (automatic) {SCR['adapted']:.3f}")
flag_tab = sm1.copy()
for nm, fl in SFLAGS1.items(): flag_tab[f"{nm}: flagged"] = fl.any(axis=1).values; flag_tab[f"{nm}: reason"] = [", ".join(fl.columns[r]) for r in fl.values]
display(flag_tab[(flag_tab["standard: flagged"]) | (flag_tab["adapted: flagged"])].round(3))

**Interpretation.** The `standard` preset applies array-GWAS rules to low-coverage data: *all or nearly all* individuals are likely to fail the 95% call-rate rule (check the table above), which would make the rule useless here, a reminder that QC thresholds must be adapted to the technology. The `adapted` preset flags only individuals that are *relatively* poor compared to the rest.

## 4.6 Remove the flagged individuals and re-scan

Variant statistics (allele frequencies, call rates, HWE) depend on which individuals are included, so after removing the individuals flagged by the `ACTIVE` preset we **re-scan the BEAGLE file** with the retained individuals, then recompute the SNP table and flags. If no individual is flagged the first scan is re-used.

In [ ]:
# 4.6  Retained individuals → second scan → final SNP table
FL_ACT = SFLAGS1[ACTIVE]
KEEP_SAMPLES = ~FL_ACT.any(axis=1).values
print(f"ACTIVE preset '{ACTIVE}': {int((~KEEP_SAMPLES).sum())} individual(s) removed, {int(KEEP_SAMPLES.sum())} retained "
      f"({(KEEP_SAMPLES & IS_CASE).sum()} {CASE_MORPH}, {(KEEP_SAMPLES & IS_CTRL).sum()} {CONTROL_MORPH})")
if (~KEEP_SAMPLES).any():
    for i in np.where(~KEEP_SAMPLES)[0]: print(f"   removed {sm1.sampleID[i]} ({sm1.morph[i]}): {', '.join(FL_ACT.columns[FL_ACT.values[i]])}")
assert KEEP_SAMPLES.sum() >= 6, "too few individuals would remain: relax the sample thresholds"
if KEEP_SAMPLES.all():
    S2, snp2 = S1, snp1
else:
    S2 = scan(np.where(KEEP_SAMPLES)[0], "retained samples")
    snp2 = snp_table(S2)
FLAGS2 = {nm: variant_flags(snp2, ps) for nm, ps in PRESETS.items()}
print(f"\nfinal scan: {len(snp2):,} SNPs × {len(S2['keep_idx'])} individuals")

In [ ]:
# 4.7  How did removing individuals change the variant QC?
fl_a, fl_b = FLAGS1[ACTIVE], FLAGS2[ACTIVE]
cmp_ = pd.DataFrame({"all individuals": fl_a.sum(), "retained individuals": fl_b.sum()})
cmp_.loc["fail ≥ 1"] = [int(fl_a.any(axis=1).sum()), int(fl_b.any(axis=1).sum())]; cmp_.loc["PASS"] = [int((~fl_a.any(axis=1)).sum()), int((~fl_b.any(axis=1)).sum())]
display(cmp_)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
cmp_.iloc[:4].plot.bar(ax=ax[0], color=[C["grey"], C["pass"]], rot=20); ax[0].set_ylabel("SNPs failing"); ax[0].set_title(f"Failing SNPs before and after sample QC ({ACTIVE})")
ax[1].hist(snp1.call_rate, bins=np.linspace(0, 1, 41), alpha=.6, color=C["grey"], label="all individuals"); ax[1].hist(snp2.call_rate, bins=np.linspace(0, 1, 41), alpha=.6, color=C["pass"], label="retained")
ax[1].set_xlabel("SNP call rate"); ax[1].legend(); ax[1].set_title("Call-rate distribution")
l1 = -np.log10(np.maximum(snp1.lrt_p_all.values, 1e-300)); l2 = -np.log10(np.maximum(snp2.lrt_p_all.values, 1e-300))
ii = rng.choice(len(l1), min(len(l1), 20000), replace=False)
ax[2].scatter(l1[ii], l2[ii], s=3, alpha=.25, color=C["dark"], rasterized=True); ax[2].plot([0, l1.max()], [0, l1.max()], "k--", lw=.8)
ax[2].set_xlabel("−log10 P HWE (all individuals)"); ax[2].set_ylabel("−log10 P HWE (retained)"); ax[2].set_title("HWE P-values change after sample QC")
plt.tight_layout(); savefig("variant_before_after_samples"); plt.show()

## 4.8 The final variant filter

With the retained individuals and the `ACTIVE` thresholds we compute the final sequential funnel (call rate → MAF/MAC → HWE → differential missingness) for **both** presets, and write the pass/fail table. This table is the answer to *"how many variants are problematic, and how many are left?"*

In [ ]:
# 4.8  Final funnel (retained individuals), both presets
def funnel_table(fl):
    keep = np.ones(len(fl), bool); rows = [("ANGSD SNPs", int(keep.sum()), 0)]
    for o in ORDER:
        before = keep.sum(); keep &= ~fl[o].values; rows.append((f"after: {o}", int(keep.sum()), int(before - keep.sum())))
    return pd.DataFrame(rows, columns=["step", "SNPs remaining", "removed at this step"]), keep
FT = {}; PASS = {}
for nm, fl in FLAGS2.items(): FT[nm], PASS[nm] = funnel_table(fl)
fin = pd.concat({nm: t.set_index("step") for nm, t in FT.items()}, axis=1); display(fin)
for nm in PRESETS: print(f"{nm:9s}: {PASS[nm].sum():>9,} of {len(snp2):,} SNPs pass ({100*PASS[nm].mean():.1f}%) ; {len(snp2)-PASS[nm].sum():,} problematic SNPs removed")

snp2["pass_standard"], snp2["pass_adapted"] = PASS["standard"], PASS["adapted"]
for nm, fl in FLAGS2.items():
    for c_ in fl.columns: snp2[f"{nm}|{c_}"] = fl[c_].values
snp2.drop(columns=["marker"]).to_csv(OUT / "qc_variants.tsv.gz", sep="\t", index=False)
sm_out = sm1.copy()
for nm, fl in SFLAGS1.items():
    for c_ in fl.columns: sm_out[f"{nm}|{c_}"] = fl[c_].values
sm_out["kept_in_final"] = KEEP_SAMPLES; sm_out.to_csv(OUT / "qc_samples.tsv", sep="\t", index=False)
FINAL_PASS = snp2[f"pass_{ACTIVE}"].values
print(f"\nwritten: qc_variants.tsv.gz, qc_samples.tsv in {OUT}")

# Part 5 — Summary figures, export, and cross-check with PLINK

## 5.1 Before and after QC

In [ ]:
# 5.1  Before / after: MAF spectrum, call rate, F, PCA
final = snp2[FINAL_PASS]; before = snp1
fig, ax = plt.subplots(2, 3, figsize=(17, 8.5))
b_ = np.linspace(0, 0.5, 51)
ax[0, 0].hist(before.maf, bins=b_, color=C["grey"], alpha=.7, label=f"before ({len(before):,})"); ax[0, 0].hist(final.maf, bins=b_, color=C["pass"], alpha=.8, label=f"after ({len(final):,})")
ax[0, 0].set_xlabel("MAF"); ax[0, 0].set_ylabel("SNPs"); ax[0, 0].set_title("Allele frequency spectrum"); ax[0, 0].legend(); panel_label(ax[0, 0], "A")
ax[0, 1].hist(before.call_rate, bins=np.linspace(0, 1, 41), color=C["grey"], alpha=.7, label="before"); ax[0, 1].hist(final.call_rate, bins=np.linspace(0, 1, 41), color=C["pass"], alpha=.8, label="after")
ax[0, 1].set_xlabel("SNP call rate"); ax[0, 1].set_title("Call rate"); ax[0, 1].legend(); panel_label(ax[0, 1], "B")
ax[0, 2].hist(before.F_all.dropna(), bins=np.linspace(-1, 1, 61), color=C["grey"], alpha=.7, label="before"); ax[0, 2].hist(final.F_all.dropna(), bins=np.linspace(-1, 1, 61), color=C["pass"], alpha=.8, label="after")
ax[0, 2].set_xlabel("F (likelihood-based)"); ax[0, 2].set_title("Departure from HWE (F)"); ax[0, 2].legend(); panel_label(ax[0, 2], "C")

tests_f = [("lrt_p_all", "GL-LRT"), ("exact_p_all", "exact (hard calls)")]
for col, lab in tests_f:
    for d, cl, nm in ((before, C["grey"], "before"), (final, C["pass"], "after")):
        qq(ax[1, 0], d[col].values, f"{lab}, {nm}", cl if lab == "GL-LRT" else (C["fail"] if nm == "before" else C["gold"]))
mx = ax[1, 0].get_xlim()[1]; ax[1, 0].plot([0, mx], [0, mx], "k--", lw=.8); ax[1, 0].set_xlabel("expected −log10 P"); ax[1, 0].set_ylabel("observed"); ax[1, 0].set_title("HWE QQ plot, before vs after"); ax[1, 0].legend(fontsize=7); panel_label(ax[1, 0], "D")

# PCA after: retained individuals and final SNPs
GRM2, w2, v2, m2 = grm_and_pca(S2, snp2, FINAL_PASS)
smk = sm1[KEEP_SAMPLES].reset_index(drop=True)
npc = min(2, len(smk) - 1)
for k_, (GRM_, w_, v_, sm_, ttl, ax_) in enumerate([(GRM1, w1, v1, sm1, "PCA before QC (all SNPs passing the lenient filter)", ax[1, 1]), (GRM2, w2, v2, smk, "PCA after QC (final SNPs, retained individuals)", ax[1, 2])]):
    pc = v_[:, :2] * np.sqrt(np.maximum(w_[:2], 0)); ev_ = 100 * np.maximum(w_, 0) / np.maximum(w_, 0).sum()
    for mo, cl in ((CASE_MORPH, C["case"]), (CONTROL_MORPH, C["ctrl"])):
        s_ = (sm_.morph == mo).values; ax_.scatter(pc[s_, 0], pc[s_, 1], s=45, color=cl, edgecolor="white", label=mo)
    ax_.set_xlabel(f"PC1 ({ev_[0]:.1f}%)"); ax_.set_ylabel(f"PC2 ({ev_[1]:.1f}%)"); ax_.set_title(ttl, fontsize=9); ax_.legend(); panel_label(ax_, "EF"[k_])
plt.tight_layout(); savefig("qc_before_after"); plt.show()
print(f"PCA after QC used {m2:,} thinned SNPs.")

## 5.2 Overall summary table — the "QC paragraph" for the paper

All numbers needed to report the QC: samples before/after, SNPs before/after, and the number failing each step for both threshold sets.

In [ ]:
# 5.2  Master summary
rows = [("individuals", N, int(KEEP_SAMPLES.sum()))]
for nm in PRESETS:
    fl = SFLAGS1[nm]
    rows.append((f"individuals flagged ({nm})", int(fl.any(axis=1).sum()), ""))
rows.append(("SNPs called by ANGSD", len(snp1), ""))
for nm in PRESETS:
    fl = FLAGS2[nm]
    for c_ in fl.columns: rows.append((f"SNPs failing '{c_}' ({nm})", int(fl[c_].sum()), f"{100*fl[c_].mean():.1f}%"))
    rows.append((f"SNPs remaining ({nm})", int(PASS[nm].sum()), f"{100*PASS[nm].mean():.1f}%"))
summary = pd.DataFrame(rows, columns=["quantity", "count", "% / retained"])
display(summary)
summary.to_csv(OUT / "qc_summary.tsv", sep="\t", index=False)

# comparison of the two preset results, as a figure
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
crit = list(FLAGS2["standard"].columns)
vals = pd.DataFrame({nm: [int(FLAGS2[nm][c_].sum()) for c_ in crit] + [int((~PASS[nm]).sum())] for nm in PRESETS}, index=crit + ["any criterion"])
vals.plot.bar(ax=ax[0], color=[C["fail"], C["gold"]], rot=20); ax[0].set_ylabel("SNPs failing"); ax[0].set_title("Problematic SNPs by criterion and threshold set")
for cont in ax[0].containers: ax[0].bar_label(cont, fmt="{:,.0f}", fontsize=7, padding=1)
sv = pd.DataFrame({nm: [int(SFLAGS1[nm][c_].sum()) for c_ in SFLAGS1[nm].columns] + [int(SFLAGS1[nm].any(axis=1).sum())] for nm in PRESETS}, index=list(SFLAGS1["standard"].columns) + ["any criterion"])
sv.plot.bar(ax=ax[1], color=[C["fail"], C["gold"]], rot=20); ax[1].set_ylabel("individuals flagged"); ax[1].set_title(f"Problematic individuals by criterion (n = {N})")
for cont in ax[1].containers: ax[1].bar_label(cont, fmt="{:,.0f}", fontsize=8, padding=1)
plt.tight_layout(); savefig("qc_summary_bars"); plt.show()

## 5.3 Export the QC-filtered data

Written to the `qc/` folder next to the ANGSD output:

| File | Content | Use |
|---|---|---|
| `HypMis_calls.QC.beagle.gz` | the BEAGLE file with only the **passing SNPs** and the **retained individuals** | `pcangsd`, ANGSD downstream analyses |
| `QC_snps_pass.txt` | `contig  position` of the passing SNPs | ANGSD `-sites` (after `angsd sites index`), PLINK `--extract` (with ids) |
| `QC_samples_keep.txt`, `QC_samples_remove.txt` | retained / removed individual IDs | PLINK `--keep`/`--remove` |
| `QC_pheno.ybin` | phenotype (1 = case, 0 = control) of the retained individuals, in file order | ANGSD association (`-yBin`) |
| `qc_variants.tsv.gz`, `qc_samples.tsv`, `qc_summary.tsv` | all QC statistics and flags | reports |

In [ ]:
# 5.3  Export the filtered BEAGLE file and the lists
pass_markers = set(snp2.marker[FINAL_PASS])
keep_idx = np.where(KEEP_SAMPLES)[0]
cols = [0, 1, 2] + [3 + 3 * i + k for i in keep_idx for k in range(3)]
out_beagle = OUT / "HypMis_calls.QC.beagle.gz"
n_out = 0
with gzip.open(out_beagle, "wt") as fo:
    for ci, ch in enumerate(pd.read_csv(BEAGLE, sep="\t", chunksize=CHUNK, dtype={"marker": str})):
        sel = ch[ch["marker"].isin(pass_markers)]
        if ci == 0: fo.write("\t".join(ch.columns[cols]) + "\n")
        if len(sel): sel.iloc[:, cols].to_csv(fo, sep="\t", header=False, index=False, float_format="%.6g"); n_out += len(sel)
print(f"{out_beagle.name}: {n_out:,} SNPs × {len(keep_idx)} individuals ({out_beagle.stat().st_size/1e6:.1f} MB)")
assert n_out == len(pass_markers), "number of exported SNPs differs from the number of passing SNPs"
snp2.loc[FINAL_PASS, ["contig", "pos"]].to_csv(OUT / "QC_snps_pass.txt", sep="\t", header=False, index=False)
(OUT / "QC_samples_keep.txt").write_text("\n".join(np.array(SAMPLES)[keep_idx]) + "\n")
(OUT / "QC_samples_remove.txt").write_text("\n".join(np.array(SAMPLES)[~KEEP_SAMPLES]) + "\n")
(OUT / "QC_pheno.ybin").write_text("\n".join(map(str, IS_CASE[keep_idx].astype(int))) + "\n")
json.dump({"active_preset": ACTIVE, "presets": PRESETS, "post_cutoff": POST_CUTOFF, "lenient_snp_call": LENIENT_SNP_CALL,
           "n_samples_in": int(N), "n_samples_out": int(KEEP_SAMPLES.sum()), "n_snps_in": int(len(snp1)), "n_snps_out": int(n_out)}, open(OUT / "qc_parameters.json", "w"), indent=2)
print("files in", OUT, ":", sorted(p.name for p in OUT.iterdir() if p.is_file() and not p.name.endswith(".pkl")))

## 5.4 Cross-check with PLINK

An independent implementation catches mistakes. We write the **hard calls** of the retained individuals to PLINK's transposed format (`.tped/.tfam`), run PLINK's own QC commands (`--missing`, `--freq`, `--hardy`, `--het`) and compare with our numbers:

* call rate: ours vs PLINK `1 − F_MISS`;
* MAF: ours (hard calls) vs PLINK `MAF`;
* HWE exact P in controls: ours vs PLINK `--hardy` row `UNAFF`.

They should match almost exactly (differences = rounding). Then we run PLINK's filters `--geno --maf --hwe` and compare the **number of SNPs removed**.

In [ ]:
# 5.4  PLINK cross-check (hard calls of the retained individuals, all SNPs of the final scan)
if PLINK is None:
    print("PLINK is not available: skipping the cross-check")
else:
    PL = OUT / "plink_check"; PL.mkdir(exist_ok=True)
    G2 = S2["G"]; a1, a2 = snp2.a1.values, snp2.a2.values; keep2 = S2["keep_idx"]
    with open(PL / "qc.tped", "w") as fh:
        for j in range(len(snp2)):
            g = G2[j]; x = np.where(g < 0, "0 0", np.where(g == 0, f"{a1[j]} {a1[j]}", np.where(g == 1, f"{a1[j]} {a2[j]}", f"{a2[j]} {a2[j]}")))
            fh.write(f"{snp2.contig[j]}\t{snp2.marker[j]}\t0\t{snp2.pos[j]}\t" + "\t".join(x) + "\n")
    with open(PL / "qc.tfam", "w") as fh:
        for i in keep2: fh.write(f"{SAMPLES[i]} {SAMPLES[i]} 0 0 0 {2 if IS_CASE[i] else 1}\n")
    r = subprocess.run(f"cd {PL} && {PLINK} --tfile qc --allow-extra-chr --missing --freq --hardy --het --out qc", shell=True, capture_output=True, text=True)
    print("\n".join(r.stdout.splitlines()[-6:]))
    lm = pd.read_csv(PL / "qc.lmiss", sep=r"\s+"); fr = pd.read_csv(PL / "qc.frq", sep=r"\s+"); hw = pd.read_csv(PL / "qc.hwe", sep=r"\s+")
    hw_u = hw[hw.TEST == "UNAFF"].set_index("SNP"); mine = snp2.set_index("marker")
    cmpdf = pd.DataFrame({"call_ours": mine.call_rate, "call_plink": 1 - lm.set_index("SNP").F_MISS,
                          "maf_ours": mine.maf_hard, "maf_plink": fr.set_index("SNP").MAF,
                          "hwe_ours": mine.exact_p_ctrl, "hwe_plink": hw_u.P}).dropna(subset=["call_plink"])
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
    ax[0].scatter(cmpdf.call_plink, cmpdf.call_ours, s=3, alpha=.2, color=C["case"], rasterized=True); ax[0].set_xlabel("PLINK call rate"); ax[0].set_ylabel("ours"); ax[0].set_title("SNP call rate")
    ax[1].scatter(cmpdf.maf_plink, cmpdf.maf_ours, s=3, alpha=.2, color=C["pass"], rasterized=True); ax[1].set_xlabel("PLINK MAF"); ax[1].set_ylabel("ours (hard calls)"); ax[1].set_title("MAF")
    v = cmpdf.dropna(subset=["hwe_plink", "hwe_ours"])
    ax[2].scatter(-np.log10(np.maximum(v.hwe_plink, 1e-300)), -np.log10(np.maximum(v.hwe_ours, 1e-300)), s=3, alpha=.2, color=C["fail"], rasterized=True); ax[2].set_xlabel("PLINK −log10 P (controls)"); ax[2].set_ylabel("ours"); ax[2].set_title("HWE exact test")
    for a in ax: a.plot(a.get_xlim(), a.get_xlim(), "k--", lw=.8)
    plt.tight_layout(); savefig("plink_crosscheck"); plt.show()
    print(f"max |difference|:  call rate {np.abs(cmpdf.call_ours-cmpdf.call_plink).max():.4f} | MAF {np.abs(cmpdf.maf_ours-cmpdf.maf_plink).max():.4f} | log10 P (HWE) {np.abs(np.log10(np.maximum(v.hwe_ours,1e-300))-np.log10(np.maximum(v.hwe_plink,1e-300))).max():.3f}")
    # filter counts: PLINK vs ours with PLINK-equivalent definitions (standard preset, hard-call MAF)
    ps = PRESETS["standard"]
    def plink_count(opts):
        rr = subprocess.run(f"cd {PL} && {PLINK} --tfile qc --allow-extra-chr {opts} --make-just-bim --out tmp", shell=True, capture_output=True, text=True)
        m_ = re.search(r"(\d+) variants remaining", rr.stdout); return int(m_.group(1)) if m_ else None
    rows = []
    for lab, opts, mine_mask in [
        (f"--geno {1-ps['snp_call']:.2f}", f"--geno {1-ps['snp_call']:.2f}", snp2.call_rate >= ps["snp_call"]),
        (f"--maf {ps['maf']}", f"--maf {ps['maf']}", snp2.maf_hard >= ps["maf"]),
        (f"--hwe {ps['hwe_p']:g} (controls)", f"--hwe {ps['hwe_p']:g}", ~(snp2.exact_p_ctrl < ps["hwe_p"]))]:
        rows.append((lab, plink_count(opts), int(mine_mask.sum())))
    display(pd.DataFrame(rows, columns=["PLINK option", "variants remaining (PLINK)", "variants remaining (ours)"]))

# Part 6 — Discussion

## 6.1 What we learned from the data

1. **Hard-call statistics are biased at low coverage.** Compare the hard-call and likelihood-based MAF, F and HWE results above: the confident genotypes are a non-random subset, which shifts MAF, heterozygote counts and the HWE test (the hard-call test is mis-calibrated). **Likelihood-based statistics** (EM allele frequency, GL HWE test) are not affected.
2. **Standard GWAS thresholds do not transfer.** A call-rate cut-off of 0.95 removes essentially all SNPs *and* all individuals. Thresholds have to be chosen from the distributions (see the sweeps) and justified.
3. **Several criteria describe the same SNPs.** UpSet plots show a large overlap between low call rate and low MAF, and HWE failures that are *not* explained by either (candidate paralogs). Sequential filtering therefore removes fewer SNPs than the sum of the individual counts.
4. **HWE has little power in a small sample**: use it for what it detects best, the *extreme* departures, and do not over-interpret it.
5. **Sample QC must be adapted to depth**: heterozygosity depends on coverage, relatedness is shrunk for low-coverage individuals, and the 6-SD PCA rule cannot trigger in a small sample.
6. **Order matters**: lenient SNP filter → sample QC → strict SNP filter; statistics are recomputed after removing individuals.

## 6.2 Pitfalls and good practice

* **Do not filter on HWE in cases** (true associations can cause departures) — or use a very lenient threshold.
* **Population structure** causes genome-wide HWE deviations (Wahlund effect). If the sample is structured, test HWE within groups or after removing the structured individuals. Sites with HWE problems in *all* groups are the genotyping errors.
* **Do not let QC differ between cases and controls**: any filter applied to only one group (e.g. HWE in controls) must be understood; test for differential missingness as we did.
* **Look before you filter**: the plots are as important as the thresholds. Document thresholds and counts (the summary table).
* **Keep the uncertainty**: after QC, prefer analyses that use genotype likelihoods/dosages (ANGSD association with `-doAsso 2`, PCAngsd) instead of hard calls.
* **QC again after imputation or after merging data sets.**

## 6.3 Suggested exercises

1. Change `POST_CUTOFF` to 0.8 and 0.99 and re-run: how do call rate, hard-call MAF and the number of HWE failures change? (`FORCE` the scan by deleting `qc/scan_all_samples.pkl`.)
2. Switch `ACTIVE` to `"standard"`. What happens? Why?
3. Try an HWE threshold of 1e-2 or 1e-8 in the `adapted` preset; use the sweep curve (3.6) to explain the result.
4. Test HWE only in cases and compare the failing sets with the controls-only test. Which SNPs differ? Are they near the M locus?
5. Which SNPs are heterozygote-excess outliers (F < −0.3)? Look at their positions: are they clustered in a few contigs (paralogs)? Do they have higher coverage than average?
6. Add the *Z chromosome* to the analysis (`INCLUDE_Z` in the association notebook) and compute heterozygosity per sample on it. What do you expect for ZW females? (*sex check*)
7. Run the association tests of the previous notebook on the QC-filtered BEAGLE file (`HypMis_calls.QC.beagle.gz`, `QC_pheno.ybin`) and compare with the unfiltered result.

## 6.4 References

* Anderson CA et al. (2010) Data quality control in genetic case-control association studies. *Nature Protocols* 5:1564.
* Marees AT et al. (2018) A tutorial on conducting genome-wide association studies: Quality control and statistical analysis. *Int J Methods Psychiatr Res* 27:e1608.
* Wigginton JE, Cutler DJ, Abecasis GR (2005) A note on exact tests of Hardy–Weinberg equilibrium. *AJHG* 76:887.
* Graffelman J, Jain D, Weir B (2017) A genome-wide study of Hardy–Weinberg equilibrium with next generation sequence data. *Human Genetics* 136:727.
* Vieira FG et al. (2016) Improving the estimation of genetic distances from NGS data: Hardy–Weinberg tests with genotype likelihoods (ANGSD/ngsTools).
* Meisner J & Albrechtsen A (2018) Inferring population structure and admixture proportions in low-depth NGS data (PCAngsd). *Genetics* 210:719.
* Manichaikul A et al. (2010) Robust relationship inference in genome-wide association studies (KING). *Bioinformatics* 26:2867.
* Yang J et al. (2011) GCTA: a tool for genome-wide complex trait analysis (GRM). *AJHG* 88:76.
* Price AL et al. (2006) Principal components analysis corrects for stratification in genome-wide association studies. *Nature Genetics* 38:904.
* Purcell S et al. (2007) PLINK. *AJHG* 81:559.